In [1]:
import numpy as np
import pandas as pd
import warnings
import re

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 20)

# ── FILE PATHS (change only here) ────────────────────────────────────────────
INPUT_CSV          = 'UJ_220726_V(220726).csv'
ROLE_SHEET         = 'Role and department sheet.xlsx'

# ── DYNAMIC OUTPUT NAME (derived from INPUT_CSV; naming only) ────────────────
# XX_DDMMYY_V(DDMMYY).csv  ->  XX_DDMMYY_Exc_V(DDMMYY)_2.xlsx  (any state code)
# When the master pipeline injects DERIVED_OUTPUT_DIR the file is written
# there; run standalone it lands next to the notebook, exactly as before.
import os as _os_dyn
_in_base = _os_dyn.path.splitext(_os_dyn.path.basename(str(INPUT_CSV)))[0]
_m_dyn = re.match(r'^([A-Za-z]+_\d+)_V\((\d+)\)$', _in_base)
_out_name = (f'{_m_dyn.group(1)}_Exc_V({_m_dyn.group(2)})_2.xlsx' if _m_dyn
             else f'{_in_base}_Exc_2.xlsx')
_out_dir = str(globals().get('DERIVED_OUTPUT_DIR', '') or '')
OUTPUT_FILE        = _os_dyn.path.join(_out_dir, _out_name) if _out_dir else _out_name
print(f'OUTPUT_FILE (dynamic) = {OUTPUT_FILE}')

# ── DYNAMIC CONSTANTS FROM INPUT FILE ───────────────────────────────────────
_cols = pd.read_csv(INPUT_CSV, nrows=0).columns.tolist()

MAX_VISITS         = max((int(m.group(1)) for c in _cols if (m := re.match(r'Visit Date_(\d+)$', c))), default=0)
MAX_CF_ASSESSMENTS = max((int(m.group(1)) for c in _cols if (m := re.search(r'_(\d+)_CF$', c))), default=0)
MAX_BF_ASSESSMENTS = max((int(m.group(1)) for c in _cols if (m := re.search(r'_(\d+)_BF$', c))), default=0)
MAX_PROTEIN        = max((int(m.group(1)) for c in _cols if (m := re.search(r'_(\d+)_P$', c))), default=0)
MAX_AN_ASSESSMENTS = max((int(m.group(1)) for c in _cols if (m := re.search(r'_(\d+)_An$', c))), default=0)

print(f'  MAX_VISITS={MAX_VISITS}, MAX_CF={MAX_CF_ASSESSMENTS}, MAX_BF={MAX_BF_ASSESSMENTS}, '
      f'MAX_PROTEIN={MAX_PROTEIN}, MAX_AN={MAX_AN_ASSESSMENTS}')

# ── REFERENCE DATE (used for gestational age on a fixed date) ─────────────
REFERENCE_DATE = pd.to_datetime('2026-06-17')

print("REFERENCE_DATE =", REFERENCE_DATE.date())

# District of THIS dataset, taken from the input file name (UJ_260726_... -> Ujjain).
# Both districts' helper sheets sit in the same Inputs folder, so the lookups
# below MUST name the district -- a wildcard would let a Jalna run fill itself
# from the Ujjain sheets. If the district cannot be determined the fills are
# skipped rather than guessed at.
HELPER_DISTRICT_BY_CODE = {'UJ': 'Ujjain', 'JL': 'Jalna', 'ML': 'Meghalaya'}
_hd_m = re.search(r'([A-Za-z]{2})_\d{6}', _os_dyn.path.basename(str(INPUT_CSV)))
HELPER_DISTRICT = HELPER_DISTRICT_BY_CODE.get(_hd_m.group(1).upper()) if _hd_m else None
HELPER_CODE = _hd_m.group(1).upper() if _hd_m else None
print(f'  Helper-sheet district: {HELPER_DISTRICT}')

# ── ROLES TO EXCLUDE FROM ANALYSIS ──────────────────────────────────────────
ROLES_TO_EXCLUDE = [
    'IIT HST Trainers', 'IITB HST Case Reviewers', 'IITB-HST Mentor',
    'NGO Staff Member', 'Mentor', 'HST Data Clerk'
]

# ── ZSCORE CHANGE CATEGORISER (reused for weight / height / WFH) ─────────────
def zscore_change_group(x, split_zero=False):
    if pd.isna(x):            return np.nan
    if split_zero and x == 0: return 'No change (0.00)'
    if x < -0.67:             return 'Faltering'
    if x > 0.67:              return 'Catchup'
    if -0.67 <= x < 0:   return '-0.67 to -0.01'
    if 0.00 <= x <= 0.67:     return '0.00 to 0.67'
    return np.nan

# ── ADOPTION DURATION BINNING ────────────────────────────────────────────────
def categorize_adoption_duration(x):
    if pd.isna(x):   return np.nan
    if x < 0:        return 'negatives'
    if x == 0:       return 'Only birth data available'
    if x <= 30:      return '01_to_30_days'
    if x <= 60:      return '31_to_60_days'
    if x <= 90:      return '61_to_90_days'
    return '91_days_or_more'

print('Config loaded')

OUTPUT_FILE (dynamic) = UJ_220726_Exc_V(220726)_2.xlsx
  MAX_VISITS=32, MAX_CF=16, MAX_BF=24, MAX_PROTEIN=13, MAX_AN=14
REFERENCE_DATE = 2026-06-17
Config loaded


In [2]:
df = pd.read_csv(INPUT_CSV)
print(f'Raw shape: {df.shape}')

df.drop_duplicates(subset=['Case ID'], inplace=True)
df.reset_index(drop=True, inplace=True)
print(f'After dedup: {df.shape}')
assert df['Case ID'].nunique() == len(df), 'Duplicate Case IDs remain!'

# ── ADD-ON (29-07-2026, revised 02-08-2026): HST-project staff ───────────────
# HST rows ('HST Trainer', '... + Calling Team Leader/Member', any HST /
# calling-team role) are NO LONGER dropped here. Per the 02-08-2026 request,
# EVERY case row stays in the derived sheet and carries its removal reason in
# the 'Reason for pre-analysis removal' column (built in the BLOCKS + REASONS
# cell near the end); the crosstab stage removes every row with a non-blank
# reason. HST rows are tagged there with reason 1, so the analysis still never
# sees them, while the derived sheet stays complete for audit.
_hst_pat0 = '|'.join([r'\bHST\b', r'calling\s*team'])
_hst_cols0 = [c for c in ['User Role', 'User Role_CR', 'Role_M'] if c in df.columns]
_hst_seen0 = pd.Series(False, index=df.index)
for _hc in _hst_cols0:
    _hst_seen0 |= df[_hc].astype(str).str.contains(_hst_pat0, case=False, na=False, regex=True)
print(f'  HST-project case row(s) found: {int(_hst_seen0.sum())}'
      f' -> KEPT in the sheet; tagged for removal in the reason column')


Raw shape: (1969, 2809)
After dedup: (1969, 2809)


In [3]:
# ── Baby Adoption Weight ──────────────────────────────────────────────────────
df['Baby Adoption Weight_CR'] = (
    df['Baby Adoption Weight_CR']
    .fillna(df['Weight_2'])
    .fillna(df["Baby's weight (in kgs)_2_CG"])
)

# ── Last Weight (uses number_of_visits to pick correct column) ───────────────
def get_last_value(row, prefix_visit, prefix_cg, count_col):
    """Return value from Visit column or CG fallback at the last visit number."""
    n = row.get(count_col)
    if pd.isna(n):
        return row.get(prefix_visit.format(1))  # fallback to visit 1 if no count
    n = int(n)
    v = row.get(prefix_visit.format(n))
    if pd.notna(v):
        return v
    return row.get(prefix_cg.format(n))

df['Last Weight_CR'] = df.apply(
    lambda r: r['Last Weight_CR'] if pd.notna(r['Last Weight_CR'])
    else get_last_value(r, 'Weight_{}', "Baby's weight (in kgs)_{}_CG", 'number_of_visits'),
    axis=1
)

# ── Baby Adoption Height ──────────────────────────────────────────────────────
df['Baby Adoption Height_CR'] = (
    df['Baby Adoption Height_CR']
    .fillna(df['Height_2'])
    .fillna(df['Length of baby (in centimeters)_2_CG'])
)

df['Last Height_CR'] = df.apply(
    lambda r: r['Last Height_CR'] if pd.notna(r['Last Height_CR'])
    else get_last_value(r, 'Height_{}', 'Length of baby (in centimeters)_{}_CG', 'number_of_visits'),
    axis=1
)

# ── Birth Weight fallback ─────────────────────────────────────────────────────
df['Birth Weight_CR'] = df['Birth Weight_CR'].fillna(df['Birth Weight (in Kgs)_C'])

print('Weight / Height / Date fallbacks applied')

Weight / Height / Date fallbacks applied


In [4]:
BIRTH_DATE_COL = 'Date of birth of baby_CR'

# ── Fill & parse DOB ──────────────────────────────────────────────────────────
df[BIRTH_DATE_COL] = (
    df[BIRTH_DATE_COL]
    .fillna(df['Date of birth_C'])
    .fillna(df['Visit Date_1'])
    .fillna(df['Measurement date_1_CG'])
)
df[BIRTH_DATE_COL] = pd.to_datetime(df[BIRTH_DATE_COL], errors='coerce')

# ── Parse all assessment dates ────────────────────────────────────────────────
assess_date_cols = [c for c in df.columns if c.startswith('Assessment date_')]
for col in assess_date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')

cf_assess_cols = [c for c in assess_date_cols if c.endswith('_CF')]
bf_assess_cols = [c for c in assess_date_cols if c.endswith('_BF')]

# ── Baby age in days at each CF / BF assessment ───────────────────────────────
for suffix, cols in [('CF', cf_assess_cols), ('BF', bf_assess_cols)]:
    for col in cols:
        num = col.split('_')[1]           # e.g. '1', '2'
        new_col = f'baby_age_{num}_calc_{suffix}'
        df[new_col] = (df[col] - df[BIRTH_DATE_COL]).dt.days

# ── Visit last ────────────────────────────────────────────────────────────────
df['Visit Date_last'] = df.apply(
    lambda r: r.get(f'Visit Date_{int(r["number_of_visits"])}')
    if pd.notna(r['number_of_visits']) else None,
    axis=1
)

# ── Last Visit Date fallback ──────────────────────────────────────────────────
def get_last_visit_date(row):
    n = row['number_of_visits']
    if pd.isna(n): n = 0
    else: n = int(n)
    vc = f'Visit Date_{n}'
    mc = f'Measurement_date_{n}_CG'
    if vc in df.columns and pd.notna(row.get(vc)):
        return row[vc]
    if mc in df.columns:
        return row.get(mc)
    return pd.NaT

df['Last Visit Date_CR'] = df.apply(get_last_visit_date, axis=1)
df['Last Visit Date_CR'] = pd.to_datetime(df['Last Visit Date_CR'], errors='coerce')

print('Date parsing & baby age columns created')

Date parsing & baby age columns created


In [5]:
VD = MAX_VISITS

# ── Visit date differences (Calculated on raw dates first) ────────────────────
for i in range(1, VD + 1):
    df[f'Visit Date_{i}'] = pd.to_datetime(df[f'Visit Date_{i}'], errors='coerce')

for i in range(1, VD):
    df[f'Visit_{i+1} - Visit_{i}'] = (
        df[f'Visit Date_{i+1}'] - df[f'Visit Date_{i}']
    ).dt.days

# ── Fill Visit Dates from CG Measurement dates after difference calculation ──
visit_nums = sorted({
    int(m.group(1))
    for col in df.columns
    if (m := re.match(r'^Visit Date_(\d+)$', col))
})
for n in visit_nums:
    vcol = f'Visit Date_{n}'
    mcol = f'Measurement date_{n}_CG'
    if vcol in df.columns and mcol in df.columns:
        df[vcol] = df[vcol].fillna(pd.to_datetime(df[mcol], errors='coerce'))

# ── Baby age at each visit (Calculated after Visit Dates are filled with fallbacks) ──
for n in visit_nums:
    vcol = f'Visit Date_{n}'
    df[f'baby_age_visit_{n}'] = (df[vcol] - df[BIRTH_DATE_COL]).dt.days

# ── Helper: consecutive numeric diff for any column prefix ───────────────────
def add_consecutive_diffs(prefix, n=VD, decimals=2):
    for i in range(1, n):
        col_a = f'{prefix}{i}'
        col_b = f'{prefix}{i+1}'
        if col_a in df.columns and col_b in df.columns:
            df[f'{prefix}{i+1} - {prefix}{i}'] = (
                (df[col_b] - df[col_a]).round(decimals) + 0.0
            )

add_consecutive_diffs('Weight_')
add_consecutive_diffs('Height_')
add_consecutive_diffs('Height Zscore_')
add_consecutive_diffs('Weight Zscore_')
add_consecutive_diffs('Percentile (W)_')
add_consecutive_diffs('Percentile (H)_')

# ── Assessment date differences (within each group: BF, CF, P, An) ────────
date_cols = [c for c in df.columns if c.startswith('Assessment date')]
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')

# Group assessment date columns by suffix (BF, CF, P, An) and diff within each
from collections import defaultdict
assess_groups = defaultdict(list)
for col in date_cols:
    suffix = col.rsplit('_', 1)[-1]  # e.g. 'BF', 'CF', 'P', 'An'
    assess_groups[suffix].append(col)

for suffix, cols in assess_groups.items():
    cols_sorted = sorted(cols, key=lambda c: int(c.split('_')[1]))  # sort by number
    for i in range(1, len(cols_sorted)):
        c1, c2 = cols_sorted[i-1], cols_sorted[i]
        df[f'{c2}-{c1}'] = (df[c2] - df[c1]).dt.days
print('Consecutive difference columns created')

Consecutive difference columns created


In [6]:
# ── Baby Adoption date ────────────────────────────────────────────────────────
df['Baby Adoption date_CR'] = (
    df['Baby Adoption date_CR']
    .fillna(df['Visit Date_2'])
    .fillna(df['Measurement date_2_CG'])
)
df['Baby Adoption date_CR'] = pd.to_datetime(df['Baby Adoption date_CR'], errors='coerce')

# ── Mother Adoption date ──────────────────────────────────────────────────────
df['Mother Adoption date_CR'] = pd.to_datetime(
    df['Mother Adoption date_CR'].fillna(df['Adoption date_M']), errors='coerce'
)

# ── Age of adoption (baby) in days ───────────────────────────────────────────
df['Age of adoption'] = (
    df['Baby Adoption date_CR'] - df[BIRTH_DATE_COL]
).dt.days

# ── Age of adoption classification ───────────────────────────────────────────
adop_bins  = [-np.inf, 0, 1, 16, 31, 61, 91, 121, 151, 181, 211, 241, 271, 301, 331, 366, np.inf]
adop_lbls  = ['invalid_age','d000','d001_to_d015','d016_to_d030','d031_to_d060',
              'd061_to_d090','d091_to_d120','d121_to_d150','d151_to_d180',
              'd181_to_d210','d211_to_d240','d241_to_d270','d271_to_d300',
              'd301_to_d330','d331_to_d365','d366_plus']
df['adoption_age_classification'] = pd.cut(
    df['Age of adoption'], bins=adop_bins, labels=adop_lbls, right=False
)

# ── Age at last visit ─────────────────────────────────────────────────────────
df['Age at Last Visit'] = (
    df['Last Visit Date_CR'] - df[BIRTH_DATE_COL]
).dt.days

# ── Duration columns ─────────────────────────────────────────────────────────
df['baby_Adoption_date - Last_visit_date'] = (
    df['Last Visit Date_CR'] - df['Baby Adoption date_CR']
).dt.days

df['Last_visit_date - mother_Adoption_date'] = (
    df['Last Visit Date_CR'] - df['Mother Adoption date_CR']
).dt.days

df['Baby_adoption_date - mother_Adoption_date'] = (
    df['Baby Adoption date_CR'] - df['Mother Adoption date_CR']
).dt.days

df['followup_duration'] = df['Age at Last Visit'] - df['Age of adoption']

# ── Adoption duration categories ──────────────────────────────────────────────
df['adoption_duration_baby']          = df['baby_Adoption_date - Last_visit_date'].apply(categorize_adoption_duration)
df['adoption_duration_mother_child']  = df['Last_visit_date - mother_Adoption_date'].apply(
    lambda x: np.nan if pd.isna(x) else
    'negatives' if x < 0 else
    'Baby not delivered yet' if x == 0 else
    '01_to_30_days' if x <= 30 else
    '31_to_60_days' if x <= 60 else
    '61_to_90_days' if x <= 90 else '91_days_or_more'
)

# ── Mother adoption ↔ birth relationship ─────────────────────────────────────
def get_diff_days(row):
    dob, adm, adb = row[BIRTH_DATE_COL], row['Mother Adoption date_CR'], row['Baby Adoption date_CR']
    if pd.isna(dob) and pd.isna(adm):  return 'DOB and Mother adoption both missing'
    if pd.isna(dob):                   return 'DOB is missing'
    if pd.isna(adm):                   return 'Mother adoption date is missing'
    if pd.isna(adb):                   return 'Baby adoption date is missing'
    diff_ad = (adb - adm).days
    if diff_ad < 0:   return 'Invalid difference: Baby Adoption date earlier than Mother adoption date'
    diff    = (dob - adm).days
    if diff > 300:    return 'Mother adoption during pregnancy is More than 300 days- Suggestive of data entry error'
    return diff

df['mother_AD_DOB'] = df.apply(get_diff_days, axis=1)

# Bin the numeric portion
m_bins = [-float('inf'), 0, 30, 60, 90, 120, 150, 180, 210, 240, 270, 300]
m_lbls = ['PNC','001_to_030','031_to_060','061_to_090','091_to_120',
          '121_to_150','151_to_180','181_to_210','211_to_240','241_to_270','271_to_300']

def bin_mother_adoption(val):
    if isinstance(val, (int, float)) and not np.isnan(val):
        return pd.cut([val], bins=m_bins, labels=m_lbls, right=True)[0]
    return val

df['Mother adoption till birth']   = df['mother_AD_DOB'].apply(bin_mother_adoption)

m_bins2 = [-float('inf'), 0, 30, 60, 90, 120, 150, 300]
m_lbls2 = ['PNC','001_to_030','031_to_060','061_to_090','091_to_120','121_to_150','More than 150 days']
df['Mother adoption till birth_2'] = df['mother_AD_DOB'].apply(
    lambda v: pd.cut([v], bins=m_bins2, labels=m_lbls2, right=True)[0]
    if isinstance(v, (int, float)) and not np.isnan(v) else v
)

print('Adoption date & duration columns created')

Adoption date & duration columns created


In [7]:
def assessment_category(x, extended=False):
    """Bin number-of-assessments into labelled groups."""
    if pd.isna(x): return '00_assessment'
    if x < 0:      return 'negatives'
    if x == 0:     return '00_assessment'
    if x == 1:     return '01_assessment'
    if x == 2:     return '02_assessments'
    if x == 3:     return '03_assessments'
    if x <= 5:     return '04_to_05_assessments'
    if not extended:
        return '06_or_more_assessments'
    if x <= 7:     return '06_to_07_assessments'
    if x <= 9:     return '08_to_09_assessments'
    return '10_or_more_assessments'

df['bf_assessment_category']   = df['number_of_assessments_BF'].apply(lambda x: assessment_category(x, extended=True))
df['bf_assessment_category_2'] = df['number_of_assessments_BF'].apply(assessment_category)
df['cf_assessment_category']   = df['number_of_assessments_CF'].apply(lambda x: assessment_category(x, extended=True))
df['cf_assessment_category_2'] = df['number_of_assessments_CF'].apply(assessment_category)

def visit_category(x):
    if pd.isna(x): return np.nan
    if x < 0:      return 'negatives'
    if x == 0:     return '00_visit'
    if x == 1:     return '01_visit'
    if x == 2:     return '02_visits'
    if x == 3:     return '03_visits'
    if x <= 5:     return '04_to_05_visits'
    if x <= 7:     return '06_to_07_visits'
    if x <= 9:     return '08_to_09_visits'
    return '10_or_more_visits'

df['visit_category'] = df['number_of_visits'].apply(visit_category)

def categorize_protein(x):
    if x == 0: return '00_assessment'
    elif x == 1: return '01_assessment'
    elif x == 2: return '02_assessments'
    elif x == 3: return '03_assessments'
    elif 4 <= x <= 5: return '04_to_05_assessments'
    elif 6 <= x <= 7: return '06_to_07_assessments'
    elif 8 <= x <= 9: return '08_to_09_assessments'
    elif x >= 10: return '10_or_more_assessments'
    else: return None  # handles NaN or negatives

df['number_of_protein_assessment_category'] = (
    pd.to_numeric(df['number_of_assessments_protein_P'], errors='coerce')
    .map(categorize_protein)
)

print('Assessment category columns created')

Assessment category columns created


In [8]:
# ── Fill Last / Adoption / Birth zscores from numbered columns ────────────────
for metric, col_prefix in [
    ('Weight Zscore', 'Last Weight Zscore_CR'),
    ('Height Zscore', 'Last Height Zscore_CR'),
    ('WFH Zscore',    'Last WFH Zscore_CR'),
]:
    df[col_prefix] = df.apply(
        lambda r, cp=col_prefix, mp=metric: r[cp] if pd.notna(r[cp])
        else r.get(f"{mp}_{int(r['number_of_visits'])}")
        if pd.notna(r.get('number_of_visits')) else np.nan,
        axis=1
    )

df['Birth Weight Zscore_CR']         = df['Birth Weight Zscore_CR'].fillna(df['Weight Zscore_1'])
df['Baby Adoption Weight Zscore_CR'] = df['Baby Adoption Weight Zscore_CR'].fillna(df['Weight Zscore_2'])
df['Birth Height Zscore_CR']         = df['Birth Height Zscore_CR'].fillna(df['Height Zscore_1'])
df['Baby Adoption Height Zscore_CR'] = df['Baby Adoption Height Zscore_CR'].fillna(df['Height Zscore_2'])
df['Birth WFH Zscore_CR']            = df['Birth WFH Zscore_CR'].fillna(df['WFH Zscore_1'])
df['Baby Adoption WFH Zscore_CR']    = df['Baby Adoption WFH Zscore_CR'].fillna(df['WFH Zscore_2'])

# ── Zscore change columns ─────────────────────────────────────────────────────
df['Last_visit_zscore_weight - adoption_zscore'] = df['Last Weight Zscore_CR'] - df['Baby Adoption Weight Zscore_CR']
df['Last_visit_zscore_height - adoption_zscore'] = df['Last Height Zscore_CR'] - df['Baby Adoption Height Zscore_CR']
df['Last_visit_zscore_weight - birth_zscore']    = df['Last Weight Zscore_CR'] - df['Birth Weight Zscore_CR']
df['Last_visit_zscore_height - birth_zscore']    = df['Last Height Zscore_CR'] - df['Birth Height Zscore_CR']

df['wfh_zscore_change_diff']        = (df['Baby Adoption WFH Zscore_CR'] - df['Birth WFH Zscore_CR']).round(2)
df['wfh_zscore_change_diff_LV_AV']  = (df['Last WFH Zscore_CR'] - df['Baby Adoption WFH Zscore_CR']).round(2)
df['wfh_zscore_change_diff_LV_DOB'] = (df['Last WFH Zscore_CR'] - df['Birth WFH Zscore_CR']).round(2)

# ── Change group categories ───────────────────────────────────────────────────
df['weight_zscore_change_group_btw_AV_BV']  = df['Weight Zscore_2 - Weight Zscore_1'].apply(zscore_change_group)
df['Height_zscore_change_group_btw_AV_BV']  = df['Height Zscore_2 - Height Zscore_1'].apply(zscore_change_group)
df['wfh_zscore_change_group_btw_AV_BV']     = df['wfh_zscore_change_diff'].apply(zscore_change_group)

df['weight_zscore_change_group_btw_LV_AV']  = df['Last_visit_zscore_weight - adoption_zscore'].apply(zscore_change_group, split_zero=True)
df['Height_zscore_change_group_btw_LV_AV']  = df['Last_visit_zscore_height - adoption_zscore'].apply(zscore_change_group, split_zero=True)
df['wfh_zscore_change_group_btw_LV_AV']     = df['wfh_zscore_change_diff_LV_AV'].apply(zscore_change_group, split_zero=True)

df['Weight_zscore_change_group_btw_LV_DOB'] = df['Last_visit_zscore_weight - birth_zscore'].apply(zscore_change_group)
df['Height_zscore_change_group_btw_LV_DOB'] = df['Last_visit_zscore_height - birth_zscore'].apply(zscore_change_group)
df['wfh_zscore_change_group_btw_LV_DOB']    = df['wfh_zscore_change_diff_LV_DOB'].apply(zscore_change_group)

# ── WFA / HFA / WFH Status classifiers ───────────────────────────────────────
def classify_wfa(z):
    if pd.isna(z): return None
    if z <= -3:    return 'SUW'
    if z <= -2:    return 'MUW'
    if z <= -1:    return 'Mild'
    return 'Normal'

def classify_hfa(z):
    if pd.isna(z): return None
    if z <= -3:    return 'SST'
    if z <= -2:    return 'MST'
    if z <= -1:    return 'Mild'
    return 'Normal'

def classify_wfh(z):
    if pd.isna(z): return None
    if z <= -3:    return 'SAM'
    if z <= -2:    return 'MAM'
    if z <= -1:    return 'Mild'
    return 'Normal'

def classify_wfa2(z):
    if pd.isna(z): return 'Zscore not available'
    if z <= -3:    return 'SUW'
    if z <= -2:    return 'MUW'
    return 'Normal'

def classify_hfa2(z):
    if pd.isna(z): return 'Zscore not available'
    if z <= -3:    return 'SST'
    if z <= -2:    return 'MST'
    return 'Normal'

def classify_wfh2(z):
    if pd.isna(z):   return 'Zscore not available'
    if z < -3:       return 'SAM'
    if -3 <= z < -2: return 'MAM'
    return 'Normal'

for suffix, bv_col, av_col, lv_col, fn in [
    ('WFA', 'Birth Weight Zscore_CR', 'Baby Adoption Weight Zscore_CR', 'Last Weight Zscore_CR', classify_wfa),
    ('HFA', 'Birth Height Zscore_CR', 'Baby Adoption Height Zscore_CR', 'Last Height Zscore_CR', classify_hfa),
    ('WFH', 'Birth WFH Zscore_CR',    'Baby Adoption WFH Zscore_CR',    'Last WFH Zscore_CR',    classify_wfh),
]:
    df[f'{suffix}_status_at_BV'] = df[bv_col].apply(fn)
    df[f'{suffix}_status_at_AV'] = df[av_col].apply(fn)
    df[f'{suffix}_status_at_LV'] = df[lv_col].apply(fn)

for suffix, bv_col, av_col, lv_col, fn in [
    ('WFA', 'Birth Weight Zscore_CR', 'Baby Adoption Weight Zscore_CR', 'Last Weight Zscore_CR', classify_wfa2),
    ('HFA', 'Birth Height Zscore_CR', 'Baby Adoption Height Zscore_CR', 'Last Height Zscore_CR', classify_hfa2),
    ('WFH', 'Birth WFH Zscore_CR',    'Baby Adoption WFH Zscore_CR',    'Last WFH Zscore_CR',    classify_wfh2),
]:
    df[f'{suffix}_status_at_BV_2'] = df[bv_col].apply(fn)
    df[f'{suffix}_status_at_AV_2'] = df[av_col].apply(fn)
    df[f'{suffix}_status_at_LV_2'] = df[lv_col].apply(fn)

# ── Extreme zscore flag ───────────────────────────────────────────────────────
df['Extreme zscore'] = np.select(
    [df['Birth Weight Zscore_CR'].isna(),
     (df['Birth Weight Zscore_CR'] < -6) | (df['Birth Weight Zscore_CR'] > 6)],
    ['No Zscore', 'Yes'], default='No'
)

print('Zscore change groups & status columns created')

Zscore change groups & status columns created


In [9]:
visit_date_cols = [c for c in df.columns
                   if c.startswith('Visit Date_') and c[len('Visit Date_'):].isdigit()]

for col in visit_date_cols:
    n = col.split('_')[-1]
    df[f'baby_days_since_adoption_{n}']  = (df[col] - df['Baby Adoption date_CR']).dt.days
    df[f'mother_days_since_adoption_{n}'] = (df[col] - df['Mother Adoption date_CR']).dt.days

# ── Classify baby age at visit ────────────────────────────────────────────────
def classify_baby_age(d):
    if pd.isna(d): return 'Unknown'
    d = int(d)
    if d == 0:       return '000_days'
    if d <= 30:      return '001_to_030_days'
    if d <= 60:      return '031_to_060_days'
    if d <= 90:      return '061_to_090_days'
    if d <= 120:     return '091_to_120_days'
    if d <= 150:     return '121_to_150_days'
    if d <= 180:     return '151_to_180_days'
    if d <= 210:     return '181_to_210_days'
    if d <= 240:     return '211_to_240_days'
    if d <= 270:     return '241_to_270_days'
    if d <= 300:     return '271_to_300_days'
    if d <= 330:     return '301_to_330_days'
    if d < 365:      return '331_to_364_days'
    return '365_or_More_days'

age_visit_cols = [c for c in df.columns if c.startswith('baby_age_visit_')]
for col in age_visit_cols:
    n = col.split('_')[-1]
    df[f'visit_age_classification_{n}'] = df[col].apply(classify_baby_age)

# ── Concatenated string columns ───────────────────────────────────────────────
def concat_cols(cols, sep='/'):
    # fillna('') before joining — prevents float NaN from slipping through
    result = (
        df[cols]
        .fillna('')                          # replace NaN with empty string first
        .astype(str)                         # then convert to string
        .apply(lambda row: sep.join(v for v in row if v not in ('', 'nan')), axis=1)
    )
    return result

df['baby_age_of_visit']       = concat_cols(age_visit_cols)
df['baby_days_from_adoption']  = concat_cols([c for c in df.columns if c.startswith('baby_days_since_adoption_')])
df['mother_days_from_adoption'] = concat_cols([c for c in df.columns if c.startswith('mother_days_since_adoption_')])

print('Days-since-adoption & string columns created')

Days-since-adoption & string columns created


In [10]:
# ── Convert weights to grams ──────────────────────────────────────────────────
weight_cols_kg = [
    c for c in df.columns
    if 'weight' in c.lower()
    and 'zscore' not in c.lower()
    and 'category' not in c.lower()
    and 'gain' not in c.lower()
    and 'measured' not in c.lower()
]
for col in weight_cols_kg:
    df[col] = pd.to_numeric(df[col], errors='coerce') * 1000

# ── Weight gain at each visit string ─────────────────────────────────────────
wt_diff_cols = [c for c in df.columns if c.startswith('Weight_') and ' - ' in c]
df['weight gain at each visit (gm)'] = df[wt_diff_cols].apply(
    lambda r: '/'.join(map(str, r.dropna())), axis=1
)

# ── Weight_last - Weight_1 ────────────────────────────────────────────────────
def weight_last_minus_first(row):
    if pd.isna(row.get('number_of_visits')): return None
    last_col = f"Weight_{int(row['number_of_visits'])}"
    if last_col not in df.columns: return None
    wl = row[last_col]
    w1 = row.get('Weight_1')
    return wl - w1 if pd.notna(wl) and pd.notna(w1) else None

df['Weight_last - Weight_1'] = df.apply(weight_last_minus_first, axis=1)
df['avg_wt_gain'] = (df['Weight_last - Weight_1'] / (df['number_of_visits'] - 1)).round(1)

# ── Gain per day between consecutive visits ───────────────────────────────────
for i in range(2, VD + 1):
    wdiff = df.get(f'Weight_{i} - Weight_{i-1}')
    vdiff = df.get(f'Visit_{i} - Visit_{i-1}')
    if wdiff is not None and vdiff is not None:
        df[f'gain_per_day_V{i}_V{i-1}'] = wdiff / vdiff

gain_per_day_cols = [c for c in df.columns if re.match(r'gain_per_day_V\d+_V\d+$', c)]
df['category_gain_per_visit'] = df[gain_per_day_cols].ge(180).any(axis=1).map(
    {True: 'above 180', False: 'less than or 180'}
)

# ── Average weight gain per day: LV vs BV and LV vs AV ───────────────────────
for label, w_start, d_start in [
    ('LV_BV', 'Birth Weight_CR',          'Date of birth of baby_CR'),
    ('LV_AV', 'Baby Adoption Weight_CR',  'Baby Adoption date_CR'),
]:
    days = (df['Last Visit Date_CR'] - pd.to_datetime(df[d_start], errors='coerce')).dt.days.replace(0, np.nan)
    df[f'Days_{label}']                    = days
    df[f'Avg_Weight_Gain_per_Day_{label}'] = ((df['Last Weight_CR'] - df[w_start]) / days).round(1)

    df[f'avg_weight_gain_28g_group_{label}'] = df[f'Avg_Weight_Gain_per_Day_{label}'].apply(
        lambda x: 'more than 28g/day' if pd.notna(x) and x > 28
        else '28 or less grams per day' if pd.notna(x) else np.nan
    )
    df[f'avg_weight_gain_17g_group_{label}'] = df[f'Avg_Weight_Gain_per_Day_{label}'].apply(
        lambda x: 'more than 17g/day' if pd.notna(x) and x > 17
        else '17 or less grams per day' if pd.notna(x) else np.nan
    )

    # Categorical bins
    bins_gpd = [-np.inf, 0, 10, 20, 30, 40, 50, np.inf]
    lbls_gpd = ['Less than 0g/d','0.1_to_10g/d','10.1_to_20g/d','20.1_to_30g/d',
                '30.1_to_40g/d','40.1_to_50g/d','More than 50g/d']
    df[f'Avg_Weight_Gain_Category_{label}'] = pd.cut(
        df[f'Avg_Weight_Gain_per_Day_{label}'], bins=bins_gpd, labels=lbls_gpd
    )

df['avg_weight_gain_LV_AV'] = (
    (df['Last Weight_CR'] - df['Baby Adoption Weight_CR']) /
    (df['Last Visit Date_CR'] - pd.to_datetime(df['Baby Adoption date_CR'], errors='coerce')).dt.days
)

# ── avg_gain_category_LV_AV (>180 check) ─────────────────────────────────────
df['avg_gain_category_LV_AV'] = np.where(
    df['avg_weight_gain_LV_AV'] > 180,
    'More than 180',
    'Less or equal to 180'
)

print('Weight gain columns created')

Weight gain columns created


In [11]:
# ── Visit scores ──────────────────────────────────────────────────────────────
def calculate_visit_scores(age, visits):
    if pd.isna(age) or pd.isna(visits): return ('NA', 'NA')
    if age < 0:   return ('Neg Age', 'Neg Age')
    if visits < 0: return ('Negative', 'Negative')
    def score(v, extended):
        if v == 1:       return 0
        if v == 2:       return 0.5
        if v == 3:       return 1
        if v <= 5:       return 2
        if v <= 7:       return 3
        if not extended: return 4
        if v <= 9:       return 4
        return 5
    if age <= 15:
        return (score(visits, False), 'NA')
    return ('NA', score(visits, True))

for col_suffix, count_col in [
    ('', 'number_of_visits'),
    (' (bf)', 'number_of_assessments_BF'),
    (' (cf)', 'number_of_assessments_CF'),
]:
    df[[f'visit score AOA less than or 15{col_suffix}',
        f'visit score AOA more than 15{col_suffix}']] = df.apply(
        lambda r, cc=count_col: pd.Series(calculate_visit_scores(r['Age of adoption'], r[cc])), axis=1
    )

# ── Birth weight category ─────────────────────────────────────────────────────
df['birthweight_category'] = df['Birth Weight_CR'].apply(
    lambda x: np.nan if pd.isna(x) else
    'negatives' if x < 0 else
    '01.5_or_less_kg' if x <= 1500 else
    '01.51_to_02.5_kg' if x <= 2500 else
    '02.51_to_02.7_kg' if x <= 2700 else
    '02.71_to_03.0_kg' if x <= 3000 else
    '03.01_to_03.5_kg' if x <= 3500 else 'More_than_03.5_kg'
)

def classify_birthweight_2(row):
    bwz = row['Birth Weight Zscore_CR']
    bwg = row['Birth Weight_CR']
    if pd.notna(bwz) and bwz < -6: return 'Below_-6SD'
    if pd.notna(bwz) and bwz >  6: return 'Above_+6SD'
    if pd.notna(bwg):
        if bwg < 1500:  return 'Less_than_01.5_kg'
        if bwg < 2500:  return '01.5_to_02.49_kg'
        if bwg < 2700:  return '02.5_to_02.69_kg'
        if bwg < 3000:  return '02.7_to_02.99_kg'
        if bwg < 3500:  return '03.0_to_03.49_kg'
        return '03.5_or_more_kg'
    return np.nan

df['birthweight_category_2'] = df.apply(classify_birthweight_2, axis=1)

df['birthweight_group'] = np.select(
    [(df['Birth Weight_CR'] >= 0) & (df['Birth Weight_CR'] < 2500),
     (df['Birth Weight_CR'] >= 2500) & (df['Birth Weight_CR'] <= 4000),
     (df['Birth Weight_CR'] > 4000)],
    ['Low Birth Weight', 'Normal Birth Weight', 'Overweight'],
    default='Birth Weight is not available'
)

# ── Activity score ────────────────────────────────────────────────────────────
df['number_of_visits'] = df['number_of_visits'] - 1
df['Activity Number'] = df[['number_of_visits',
                             'number_of_assessments_BF',
                             'number_of_assessments_CF']].sum(axis=1)

def classify_activity_score(x):
    if pd.isna(x): return 'NA'
    elif x < 0:      return 'NEG'
    elif 0 < x <= 2:     return '01_to_02'
    elif 3 < x <= 10:    return '03_to_10'
    elif 11 < x <= 20:    return '11_to_20'
    elif 21 < x <= 30:    return '21_to_30'
    return '31_or_above'

df['Activity Score'] = df['Activity Number'].apply(classify_activity_score)

# ── Transition label & score ──────────────────────────────────────────────────
stage_map = {'Faltering': 1, '-0.67 to -0.01': 2, '0.00 to 0.67': 3, 'Catchup': 4}
transition_map = {
    ('Faltering', '-0.67 to -0.01'): 'Faltering -> (-0.67 to -0.01)',
    ('Faltering', '0.00 to 0.67'):   'Faltering -> (0.00 to 0.67)',
    ('Faltering', 'Catchup'):        'Faltering -> (Catchup)',
    ('-0.67 to -0.01', 'Faltering'): '(-0.67 to -0.01) -> Faltering',
    ('-0.67 to -0.01', '0.00 to 0.67'): '(-0.67 to -0.01) -> (0.00 to 0.67)',
    ('-0.67 to -0.01', 'Catchup'):   '(-0.67 to -0.01) -> (Catchup)',
    ('0.00 to 0.67', 'Faltering'):   '(0.00 to 0.67) -> Faltering',
    ('0.00 to 0.67', '-0.67 to -0.01'): '(0.00 to 0.67) -> (-0.67 to -0.01)',
    ('0.00 to 0.67', 'Catchup'):     '(0.00 to 0.67)-> (Catchup)',
    ('Catchup', 'Faltering'):        '(Catchup) -> Faltering',
    ('Catchup', '0.00 to 0.67'):     '(Catchup) -> (0.00 to 0.67)',
    ('Catchup', '-0.67 to -0.01'):   '(Catchup) -> (-0.67 to -0.01)',
}

def get_transition(row):
    a, b = row['weight_zscore_change_group_btw_AV_BV'], row['weight_zscore_change_group_btw_LV_AV']
    if b == 'No change (0.00)': return 'No change (0.00)'
    if a == b: return 'Same Status'
    return transition_map.get((a, b), 'Unknown Transition')

df['transition_label'] = df.apply(get_transition, axis=1)
def get_transition_score(row):
    if row["transition_label"] == "Same Status":
        return 0
    
    stage_from = stage_map.get(row["weight_zscore_change_group_btw_AV_BV"], None)
    stage_to = stage_map.get(row["weight_zscore_change_group_btw_LV_AV"], None)
    
    if stage_from is not None and stage_to is not None and pd.notna(stage_from) and pd.notna(stage_to):
        return stage_to - stage_from
    return np.nan

df['Transition Score'] = df.apply(get_transition_score, axis=1)

# ── LV-AV classification ──────────────────────────────────────────────────────
df['LV_AV_classification'] = df['baby_Adoption_date - Last_visit_date'].apply(
    lambda x: np.nan if pd.isna(x) else
    'negatives' if x < 0 else '00_days' if x == 0 else
    '01_to_30_days' if x <= 30 else '31_to_60_days' if x <= 60 else
    '61_to_90_days' if x <= 90 else '91_days_or_more'
)

# ── WFA combo column ──────────────────────────────────────────────────────────
df['WFA_at_LV/Change_zscore_LV_AV'] = (
    df['WFA_status_at_LV'].astype(str) + '/' +
    df['weight_zscore_change_group_btw_LV_AV'].astype(str)
)

# ── Last visit age classification ─────────────────────────────────────────────
lv_bins1 = [0, 15, 30, 60, 90, 120, 150, 180, 210, 240, 270, 300, 330, 365, np.inf]
lv_lbls1 = ['d001_to_d015','d016_to_d030','d031_to_d060','d061_to_d090','d091_to_d120',
            'd121_to_d150','d151_to_d180','d181_to_d210','d211_to_d240','d241_to_d270',
            'd271_to_d300','d301_to_d330','d331_to_d365','d366_plus']
df['last visit age classification_1'] = pd.cut(df['Age at Last Visit'], bins=lv_bins1, labels=lv_lbls1, right=True)

lv_bins2 = [0, 60, 120, 180, 210, 240, 270, 300, 330, 360, np.inf]
lv_lbls2 = ['d001_to_d060','d061_to_d120','d121_to_d180','d181_to_d210','d211_to_d240',
            'd241_to_d270','d271_to_d300','d301_to_d330','d331_to_d360','d361_plus']
df['last visit age classification_2'] = pd.cut(df['Age at Last Visit'], bins=lv_bins2, labels=lv_lbls2, right=True)

print('Visit scores, birth weight, activity & transition columns created')

Visit scores, birth weight, activity & transition columns created


In [12]:
# ── Gestational age ───────────────────────────────────────────────────────────
df['Last Menstrual Period (LMP)_M'] = pd.to_datetime(df['Last Menstrual Period (LMP)_M'], errors='coerce')
df['Last Menstrual Period (LMP)_CR'] = pd.to_datetime(df['Last Menstrual Period (LMP)_CR'], errors='coerce')
lmp_col = df['Last Menstrual Period (LMP)_M'].fillna(df['Last Menstrual Period (LMP)_CR'])

df['gestational_weeks'] = ((df[BIRTH_DATE_COL] - lmp_col).dt.days / 7).round(1)

ga_bins  = [0, 20, 28, 34, 36.9, 41.9, float('inf')]
ga_lbls  = ['20 weeks or less','20.1 to 28 weeks','28.1 to 34 weeks',
            '34.1 to 36.9 weeks','37 to 41.9 weeks','42 weeks or more']
df['gestational week classification'] = pd.cut(df['gestational_weeks'], bins=ga_bins, labels=ga_lbls)

day = REFERENCE_DATE.day

if 11 <= day <= 13:
    suffix = "th"
else:
    suffix = {1: "st", 2: "nd", 3: "rd"}.get(day % 10, "th")

gest_col_name = (
    f"gestational_age_on_{day}{suffix}_{REFERENCE_DATE.strftime('%b')}"
)

lmp_m_only = pd.to_datetime(df['Last Menstrual Period (LMP)_M'], errors='coerce')
df[gest_col_name] = lmp_m_only.apply(
    lambda x: (REFERENCE_DATE - x).days if pd.notna(x) else 'No LMP'
)

df['gestational_age_category'] = df[gest_col_name].apply(
    lambda v: 'gestational_age>=280' if isinstance(v, (int, float)) and v >= 280
    else 'gestational_age<280' if isinstance(v, (int, float)) else 'No Value'
)

# ── Protein assessment columns — pregnancy duration ───────────────────────────
lmp = pd.to_datetime(df['Last Menstrual Period (LMP)_M'], errors='coerce')
df['Last Menstrual Period (LMP)_M'] = lmp
for n in range(1, MAX_PROTEIN + 1):
    ac = f'Assessment date_{n}_P'
    if ac not in df.columns: continue
    df[ac] = pd.to_datetime(df[ac], errors='coerce')
    df[f'Pregnancy duration during protein count_{n}_weeks'] = np.where(
        lmp.isna(), 'No LMP',
        ((df[ac] - lmp).dt.days / 7).round(1)
    )

pg_bins = [0, 12, 14, 16, 18, 20, 22, 24, 26, 28, 30, 32, 34, 36, 38, 40, 42, float('inf')]
pg_lbls = ['12 weeks or less','12.1 to 14 weeks','14.1 to 16 weeks','16.1 to 18 weeks',
           '18.1 to 20 weeks','20.1 to 22 weeks','22.1 to 24 weeks','24.1 to 26 weeks',
           '26.1 to 28 weeks','28.1 to 30 weeks','30.1 to 32 weeks','32.1 to 34 weeks',
           '34.1 to 36 weeks','36.1 to 38 weeks','38.1 to 40 weeks','40.1 to 42 weeks','42 weeks above']
for n in range(1, MAX_PROTEIN + 1):
    dc = f'Pregnancy duration during protein count_{n}_weeks'
    cc = f'Pregnancy duration during protein count_{n}_weeks_classification'
    if dc not in df.columns: continue
    df[cc] = df[dc].apply(lambda x: 'No LMP' if str(x).strip().lower() == 'no lmp' else np.nan)
    df[dc] = pd.to_numeric(df[dc], errors='coerce')
    df.loc[df[dc].notna(), cc] = pd.cut(df.loc[df[dc].notna(), dc], bins=pg_bins, labels=pg_lbls)

# ── Merge protein-combined columns ───────────────────────────────────────────
# 1. Pregnancy duration (weeks)
p_dur_cols = [f'Pregnancy duration during protein count_{i}_weeks' for i in range(1, MAX_PROTEIN+1) if f'Pregnancy duration during protein count_{i}_weeks' in df.columns]
df['Pregnancy duration during protein count (weeks)'] = df[p_dur_cols].apply(
    lambda r: '/'.join(r.dropna().astype(str)), axis=1
)

# 2. Pregnancy duration (weeks) classification
p_class_cols = [f'Pregnancy duration during protein count_{i}_weeks_classification' for i in range(1, MAX_PROTEIN+1) if f'Pregnancy duration during protein count_{i}_weeks_classification' in df.columns]
def combine_classifications(row):
    values = row.dropna().astype(str).tolist()
    if 'No LMP' in values:
        # Keep 'No LMP' once at the start
        values = ['No LMP'] + [v for v in values if v != 'No LMP']
    return '/'.join(values) if values else np.nan

df['Pregnancy duration during protein count (weeks) classification'] = df[p_class_cols].apply(combine_classifications, axis=1)

# 3. Protein count (grams) columns
for col_base, out_name in [
    ('Total protein from all foods (grams)_{}_P', 'Total protein from all foods (grams)'),
    ('Excellent and high quality protein (grams)_{}_P', 'Excellent and high quality protein (grams)'),
]:
    existing = [col_base.format(i) for i in range(1, MAX_PROTEIN+1) if col_base.format(i) in df.columns]
    if existing:
        df[out_name] = df[existing].apply(
            lambda r: '/'.join(r.dropna().astype(str)), axis=1
        )

# ── Delivery method cleanup ───────────────────────────────────────────────────
df['Method of delivery_C'] = df['Method of delivery_C'].replace(
    {'C-Section': 'Cesarean Section', 'Cesarean Operation': 'Cesarean Section'}
)

# ── Mother age group ──────────────────────────────────────────────────────────
def categorize_age(a):
    if pd.isna(a): return np.nan
    if 15 <= a <= 18: return '15_to_18_years_old'
    if 19 <= a <= 24: return '19_to_24_years_old'
    if 25 <= a <= 36: return '25_to_36_years_old'
    if 37 <= a <= 46: return '37_to_46_years_old'
    if 47 <= a <= 50: return '47_to_50_years_old'
    return np.nan

df['Mother_Age_Group'] = df["Mother's Age (years)_M"].apply(categorize_age)

# ── Location of delivery ──────────────────────────────────────────────────────
def categorize_delivery(loc):
    if pd.isna(loc): return np.nan
    if loc == 'Home': return 'Home'
    if 'Private' in str(loc): return 'Private'
    return 'Government'

df['Location_of_Delivery_category'] = df['Location of delivery_C'].apply(categorize_delivery)

# List of variable prefixes to merge
variable_prefixes = [
    "Does this child have any illness?",
    "Illness observed",
    "Was child fed anything except breastmilk?",
    "Food given to the baby",
    "Type details of other food"
]

# Create new merged columns
for var in variable_prefixes:
    matching_cols = [col for col in df.columns if col.startswith(var + "_")]

    if not matching_cols:
        continue

    df[var + "_merged"] = df[matching_cols].apply(
        lambda row: " / ".join(map(str, row)),
        axis=1
    )

# Display the updated DataFrame
df.head()

# ── Other food counts ─────────────────────────────────────────────────────────
ff_col = 'Was child fed anything except breastmilk?_merged'
if ff_col in df.columns:
    df[ff_col] = df[ff_col].astype(str).str.lower().str.strip()
    df['Other food (yes,no both)']     = df[ff_col].apply(
        lambda x: sum(1 for v in x.split(' / ') if v in ('yes','no')))
    df['Other food (only yes each visit)'] = df[ff_col].apply(
        lambda x: sum(1 for v in x.split(' / ') if v == 'yes'))

print('Gestational age, delivery & merged columns created')

Gestational age, delivery & merged columns created


In [13]:
# ── Load mapping sheet ────────────────────────────────────────────────────────
mapping_df = pd.read_excel(ROLE_SHEET, sheet_name='Sheet1')

# ADD-ON (29-07-2026): the raw exports write some role values with stray
# leading/trailing spaces (e.g. 'ASHA SUPERVISOR + Activity Mentor ') and a few
# mapping-sheet keys carry them too, so the exact-match lookups below silently
# dropped real roles into Role Group 'Others' and Department 'z_NA' -- which is
# why the Department rows of every crosstab did not add up to the Total.
# Whitespace is now normalised on BOTH sides before the lookup: leading/trailing
# stripped, internal runs collapsed to one space. Matching is otherwise
# unchanged (still case-sensitive). Verified on this sheet: 186 distinct keys
# before and after normalising -- no key collisions, no remapping.
def _norm_role(v):
    return ' '.join(str(v).split())

role_map   = {_norm_role(k): v for k, v in zip(mapping_df['role'], mapping_df['Role group'])}
dept_map   = {_norm_role(k): v for k, v in zip(mapping_df['role'], mapping_df['Department'])}

# ── ONE-TIME DOCS ROLE FILL (29-07-2026) ─────────────────────────────────────
# A number of learners have NO role at all in any of the three raw role columns,
# so they end up as Role Group 'z_NA' and drop out of the analysis. The DOCS
# skill-refreshment sheet records the role for many of them. This add-on fills
# the RAW role column for those learners only -- rows that already carry a role
# are never touched -- so the normal mapping below then classifies them exactly
# like any other case.
# ONE-TIME: driven by Inputs\DOCS_onetime_fill_<District>.xlsx. If that file is
# not present this block prints a line and does nothing, so later datasets are
# unaffected. Set DOCS_ONE_TIME_ROLE_FILL = False to disable it outright.
DOCS_ONE_TIME_ROLE_FILL = True
DOCS_FILL_STATS = {}

if not DOCS_ONE_TIME_ROLE_FILL:
    print('  DOCS_ONE_TIME_ROLE_FILL is False -> no role fill')
else:
    import glob as _glob_docs
    _dk_dir = _os_dyn.path.dirname(_os_dyn.path.abspath(str(ROLE_SHEET)))
    _dk_hits = ([] if not HELPER_DISTRICT else _glob_docs.glob(
        _os_dyn.path.join(_dk_dir, f'DOCS_onetime_fill_{HELPER_DISTRICT}.xlsx')))
    if not HELPER_DISTRICT:
        print('  District could not be determined from INPUT_CSV ->'
              ' DOCS role fill skipped (never guess the district)')
    if not _dk_hits:
        print('  DOCS one-time fill sheet not found -> role fill skipped (normal for a new dataset)')
    else:
        _dk_path = max(_dk_hits, key=_os_dyn.path.getmtime)
        _dk = pd.read_excel(_dk_path)
        _dk_lut = {str(k).strip(): v for k, v in zip(_dk['User Acc ID'], _dk['Role'])
                   if isinstance(v, str) and v.strip()}
        _dk_idcols = [c for c in ['User Acc ID', 'User Acc ID_CR', 'User Acc ID_BF',
                                  'User Acc ID_CF', 'User Acc ID_C', 'User Acc ID_M',
                                  'User Acc ID_P', 'User Acc ID_An', 'User Acc ID_CG']
                      if c in df.columns]
        _dk_rolecols = [c for c in ['User Role', 'User Role_CR', 'Role_M'] if c in df.columns]
        if not _dk_idcols or not _dk_rolecols:
            print('  DOCS role fill: required ID / role columns missing -> skipped')
        else:
            _dk_id = (df[_dk_idcols].bfill(axis=1).iloc[:, 0]
                      .astype(str).str.strip().str.replace(r'\.0$', '', regex=True))
            _dk_blank = df[_dk_rolecols].isna().all(axis=1)
            _dk_found = _dk_id.where(_dk_blank).map(_dk_lut)
            _dk_hit = _dk_found.notna()
            _dk_lrn0 = _dk_id[_dk_blank].nunique()
            df.loc[_dk_hit, _dk_rolecols[0]] = _dk_found[_dk_hit]
            _dk_lrn1 = _dk_id[df[_dk_rolecols].isna().all(axis=1)].nunique()
            DOCS_FILL_STATS = {'role rows blank before': int(_dk_blank.sum()),
                               'role rows filled': int(_dk_hit.sum()),
                               'learners blank before': int(_dk_lrn0),
                               'learners resolved': int(_dk_lrn0) - int(_dk_lrn1)}
            print(f'  DOCS role fill from {_os_dyn.path.basename(_dk_path)}:'
                  f' filled {int(_dk_hit.sum())} of {int(_dk_blank.sum())} role-less case row(s)'
                  f' ({int(_dk_lrn0) - int(_dk_lrn1)} of {int(_dk_lrn0)} learner(s) resolved)')
            if _dk_hit.any():
                print(f'  Roles supplied by DOCS: {dict(_dk_found[_dk_hit].value_counts())}')

# ── MASD ROLE FILL (02-08-2026) ──────────────────────────────────────────────
# Second role gap-filler, after the DOCS fill: learners whose raw role columns
# are ALL blank get their role from the MASD sheet ('Role', keyed on 'User
# Account ID'). Per the 02-08-2026 rule, a case only counts as 'no role' if the
# role is missing even after checking BOTH the MASD and DOCS sheets. Only
# still-blank rows are touched; rows that already carry a role are never
# changed. District-aware; skipped loudly if the MASD file cannot be found.
MASD_ROLE_FILL = True
if not MASD_ROLE_FILL:
    print('  MASD_ROLE_FILL is False -> no MASD role fill')
elif not HELPER_DISTRICT:
    print('  District could not be determined -> MASD role fill skipped (never guess)')
else:
    import glob as _glob_mr
    _mr_dir = _os_dyn.path.dirname(_os_dyn.path.abspath(str(ROLE_SHEET)))
    _mr_hits = []
    for _mr_pat in ([f'{HELPER_CODE}_raw_combined_*_MASD.xlsx'] if HELPER_CODE else []) \
            + [f'MASD_{HELPER_DISTRICT}_*_combined.xlsx']:
        for _mr_d in (_mr_dir, _os_dyn.path.dirname(_mr_dir)):
            _mr_hits += _glob_mr.glob(_os_dyn.path.join(_mr_d, _mr_pat))
        if _mr_hits:
            break
    _mr_hits = [h for h in _mr_hits if not _os_dyn.path.basename(h).startswith('~$')]
    if not _mr_hits:
        print('  MASD sheet not found -> MASD role fill skipped')
    else:
        _mr_path = max(_mr_hits, key=_os_dyn.path.getmtime)
        _mr = pd.read_excel(_mr_path)
        _mr_lut = {str(k).strip(): v for k, v in zip(_mr['User Account ID'], _mr['Role'])
                   if isinstance(v, str) and v.strip()}
        _mr_idcols = [c for c in ['User Acc ID', 'User Acc ID_CR', 'User Acc ID_BF',
                                  'User Acc ID_CF', 'User Acc ID_C', 'User Acc ID_M',
                                  'User Acc ID_P', 'User Acc ID_An', 'User Acc ID_CG']
                      if c in df.columns]
        _mr_rolecols = [c for c in ['User Role', 'User Role_CR', 'Role_M'] if c in df.columns]
        if not _mr_idcols or not _mr_rolecols:
            print('  MASD role fill: required ID / role columns missing -> skipped')
        else:
            _mr_id = (df[_mr_idcols].bfill(axis=1).iloc[:, 0]
                      .astype(str).str.strip().str.replace(r'\.0$', '', regex=True))
            _mr_blank = df[_mr_rolecols].isna().all(axis=1)
            _mr_found = _mr_id.where(_mr_blank).map(_mr_lut)
            _mr_hit = _mr_found.notna()
            _mr_l0 = _mr_id[_mr_blank].nunique()
            df.loc[_mr_hit, _mr_rolecols[0]] = _mr_found[_mr_hit]
            _mr_l1 = _mr_id[df[_mr_rolecols].isna().all(axis=1)].nunique()
            print(f'  MASD role fill from {_os_dyn.path.basename(_mr_path)}:'
                  f' filled {int(_mr_hit.sum())} of {int(_mr_blank.sum())} role-less case row(s)'
                  f' ({int(_mr_l0) - int(_mr_l1)} of {int(_mr_l0)} learner(s) resolved)')
            if _mr_hit.any():
                print(f'  Roles supplied by MASD: {dict(_mr_found[_mr_hit].value_counts())}')

# ── Role Group ────────────────────────────────────────────────────────────────
def classify_role(row):
    for col in ['User Role', 'User Role_CR', 'Role_M']:
        v = row.get(col)
        if pd.notna(v) and _norm_role(v) in role_map: return role_map[_norm_role(v)]
    if pd.isna(row.get('User Role')) and pd.isna(row.get('User Role_CR')) and pd.isna(row.get('Role_M')):
        return 'z_NA'
    # 02-08-2026: the sheet's catch-all group is 'Other' (singular); the old
    # 'Others' fallback created a second, separate bucket in every table
    return 'Other'

df['Role Group'] = df.apply(classify_role, axis=1)

# ── Department ────────────────────────────────────────────────────────────────
def classify_department(row):
    for col in ['User Role', 'User Role_CR', 'Role_M', 'Role Group']:
        v = row.get(col)
        if pd.notna(v) and _norm_role(v) in dept_map: return dept_map[_norm_role(v)]
    return 'z_NA'

df['Department'] = df.apply(classify_department, axis=1)

# ── Learner category ──────────────────────────────────────────────────────────
df['Learner Category_2'] = df['Member Tag_CR'].apply(
    lambda t: 'MT + FL' if pd.notna(t) and str(t).strip() in ['Master Trainer','Facilitator']
    else 'Other'
)

# ── Learner category 3 ──────────────────────────────────────────────────────
# NA / blank / any spelling of "under evaluation" -> 'Under Evaluation'
# Master Trainer / Facilitator                    -> 'MT + FL'
# everything else                                 -> 'Other'  (unified 02-08-2026)
def classify_learner_category_3(t):
    if pd.isna(t) or str(t).strip().lower() in ['', 'na', 'n/a', 'nan']:
        return 'Under Evaluation'
    if re.sub(r'[^a-z]', '', str(t).lower()) == 'underevaluation':
        return 'Under Evaluation'
    if str(t).strip() in ['Master Trainer', 'Facilitator']:
        return 'MT + FL'
    return 'Other'

df['Learner Category_3'] = df['Member Tag_CR'].apply(classify_learner_category_3)

# ── Consolidate multi-source ID / name columns ────────────────────────────────
def consolidate(df, sources, new_col, drop=True):
    existing = [c for c in sources if c in df.columns]
    df[new_col] = df[existing].bfill(axis=1).iloc[:, 0] if existing else np.nan
    if drop:
        df.drop(columns=[c for c in existing if c != new_col], inplace=True, errors='ignore')

consolidate(df,
    ['User Acc ID','User Acc ID_CR','User Acc ID_BF','User Acc ID_CF',
     'User Acc ID_C','User Acc ID_M','User Acc ID_P','User Acc ID_An','User Acc ID_CG'],
    'User Acc ID_all')

consolidate(df,
    ['User Name','User Name_BF','User Name_CF','User Name_C','User Name_M',
     'User Name_P','User Name_An','User Name_CG'],
    'User Name_all')

consolidate(df,
    ['Facility/NGO','NGO/Facility_BF','NGO/Facility_CF','NGO/Facility_C',
     'NGO/Facility_CG','NGO/Facility_M','NGO/Facility_P','NGO/Facility_An'],
    'Facility/NGO_all')

consolidate(df, ['Child ID','Child ID_CR','Child ID_CG'], 'Child ID_all')
consolidate(df, ['Name of mother','Name of mother_CR',"Mother's Name_M"], 'Mother Name_all')
consolidate(df, ['Child Name','Child Name_CR'], 'Child Name_all')
consolidate(df, ['Child Gender','Gender of baby_CR',"Baby's gender_C",'Child Gender_CG'], 'Gender of Baby')

consolidate(df,
    ['User Reg Id_BF','User Reg Id_CF','User Reg Id_C','User Reg Id_M',
     'User Reg Id_P','User Reg Id_An','User Reg Id_CG','Reg ID','Reg ID_CR'],
    'User Reg Id_all')

# ── Number of adoptions per user ──────────────────────────────────────────────
df['number_of_adoption'] = df.groupby('User Acc ID_all')['User Acc ID_all'].transform('count')

def categorize_adoption_count(n):
    if pd.isna(n) or n < 1: return np.nan
    if n <= 3:  return '01_to_03'
    if n <= 6:  return '04_to_06'
    if n <= 9:  return '07_to_09'
    return '10_or_more'

df['Total Adoptions group'] = df['number_of_adoption'].apply(categorize_adoption_count)

# ── Jalna adoption grouping ──────────────────────────────────────────────────
def categorize_adoption_count_jalna(n):
    if pd.isna(n) or n < 1: return np.nan
    if n == 1: return '01'
    if n == 2: return '02'
    if n == 3: return '03'
    return 'More_than_03'

df['Total Adoptions group_jalna'] = df['number_of_adoption'].apply(categorize_adoption_count_jalna)

# ── Expected adoptions (3 separate columns) ───────────────────────────────────
# Nursing staff: 1 ANC, 3 Less than 5 month, 1 More than 5 month
# (the mapping sheet renamed this role group 'Staff nurse' -> 'Nursing staff', 29-07-2026)
# All others:  1 ANC, 1 Less than 5 month, 1 More than 5 month
df['Expected adoptions ANC'] = 1

df['Expected adoptions Less than 5 month'] = df['Role Group'].apply(
    lambda x: 3 if x == 'Nursing staff' else 1
)

df['Expected adoptions More than 5 month'] = 1


print('Role / department / consolidated ID columns created')

Role / department / consolidated ID columns created


In [14]:
district_sources = ['District','District_CR','District_CG','District_BF','District_CF',
                    'District_C','District or similar region_M','District_P',
                    'Phc District_CR','District_An']
existing_dist = [c for c in district_sources if c in df.columns]

df['District_all'] = (
    df[existing_dist].bfill(axis=1).iloc[:, 0]
    .astype(str).str.strip().str.title()
)

def clean_district(col):
    return (
        col.astype(str)
        .str.lower()
        .str.replace(r"mcj|mother'?s", '', regex=True)
        .str.replace(r'\s+', ' ', regex=True)
        .str.strip()
        .str.title()
    )

df['District_all'] = clean_district(df['District_all'])
df.drop(columns=[c for c in existing_dist if c in df.columns and c != 'District or similar region_M'], inplace=True, errors='ignore')
df.rename(columns={'District_all': 'District'}, inplace=True)

print('District consolidated Unique values:', df['District'].nunique())

District consolidated Unique values: 1


In [15]:
# ── Row-level impact flags ────────────────────────────────────────────────────
df['impact_child_YN_DP'] = df['weight_zscore_change_group_btw_LV_AV'].apply(
    lambda x: 'Yes' if x == 'Catchup' else 'No'
)

df['impact_child_YN_RD'] = df.apply(
    lambda r: 'Data is missing' if pd.isna(r.get('Last Weight Zscore_CR'))
    else 'Yes' if (r.get('weight_zscore_change_group_btw_LV_AV') == 'Catchup'
                  and r.get('Last Weight Zscore_CR') >= -1)
    else 'No', axis=1
)

df['impact_child_YN_RD_2'] = df.apply(
    lambda r: 'Data is missing' if pd.isna(r.get('Last Weight Zscore_CR'))
    else 'Yes' if (r.get('weight_zscore_change_group_btw_LV_AV') == 'Catchup'
                  and r.get('Last Weight Zscore_CR') >= -2)
    else 'No', axis=1
)

# ── Learner-level counts ──────────────────────────────────────────────────────
def add_impact_count(filter_condition, col_name):
    counts = (
        df[filter_condition]
        .groupby('User Acc ID_all').size()
        .reset_index(name=col_name)
    )
    return df.merge(counts, on='User Acc ID_all', how='left').fillna({col_name: 0})

df = add_impact_count(
    df['weight_zscore_change_group_btw_LV_AV'] == 'Catchup',
    'impact_child_count_DP'
)
df['impact_child_count_DP'] = df['impact_child_count_DP'].astype(int)

df = add_impact_count(
    (df['Last Weight Zscore_CR'] >= -1) & (df['weight_zscore_change_group_btw_LV_AV'] == 'Catchup'),
    'impact_child_count_RD'
)
df['impact_child_count_RD'] = df['impact_child_count_RD'].astype(int)

df = add_impact_count(
    (df['Last Weight Zscore_CR'] >= -2) & (df['weight_zscore_change_group_btw_LV_AV'] == 'Catchup'),
    'impact_child_count_RD_2'
)
df['impact_child_count_RD_2'] = df['impact_child_count_RD_2'].astype(int)

# ── Percentages ───────────────────────────────────────────────────────────────
for cnt, pct in [
    ('impact_child_count_RD',   'Total_impact_children_for_learner_%_RD'),
    ('impact_child_count_RD_2', 'Total_impact_children_for_learner_%_RD_2'),
    ('impact_child_count_DP',   'Total_impact_children_for_learner_%_DP'),
]:
    df[pct] = (df[cnt] / df['number_of_adoption'] * 100).round(1)

# ── Binned group columns ──────────────────────────────────────────────────────
cnt_bins  = [-1, 0, 2, 4, 6, 8, 10, float('inf')]
cnt_lbls  = ['0','01_to_02','03_to_04','05_to_06','07_to_08','09_to_10','more_than_10']
pct_bins  = [-1, 0, 20, 40, 60, 80, 100]
pct_lbls  = ['0','01_to_20','21_to_40','41_to_60','61_to_80','81_to_100']

for suffix in ['RD', 'RD_2', 'DP']:
    cnt_col = f'impact_child_count_{suffix}'
    pct_col = f'Total_impact_children_for_learner_%_{suffix}'
    grp_cnt = f'impact_child_{suffix}_group'
    grp_pct = f'impact_children_%_{suffix}_group'

    df[grp_cnt] = pd.cut(df[cnt_col], bins=cnt_bins, labels=cnt_lbls)
    df[grp_cnt] = df[grp_cnt].cat.add_categories(['No impact count/child'])
    df.loc[df[cnt_col] == 0, grp_cnt] = 'No impact count/child'

    df[grp_pct] = pd.cut(df[pct_col], bins=pct_bins, labels=pct_lbls, include_lowest=True)
    df[grp_pct] = df[grp_pct].cat.add_categories(['No impact count/child'])
    df.loc[df[pct_col] == 0, grp_pct] = 'No impact count/child'

print('Impact child count & percentage columns created')

Impact child count & percentage columns created


In [16]:
# ── Adoption type columns ─────────────────────────────────────────────────────
def type_of_adoption(row):
    adoption = row['Mother adoption till birth']
    age      = row['Age of adoption']
    if pd.isna(adoption) and pd.isna(age):
        return 'Both Mother adoption and child age are missing'
    if pd.isna(adoption): return 'Mother adoption date is missing'
    if pd.isna(age):      return 'Data to calculate child age of adoption is not available'
    if pd.notna(age) and age < 0:
        return 'Age of adoption of the child is negative suggestive of data entry error'
    if adoption == 'PNC':
        return 'PNCL5M' if 0 <= age <= 150 else 'PNCG5M'
    if adoption in ['001_to_030','031_to_060','061_to_090','091_to_120',
                    '121_to_150','151_to_180','181_to_210','211_to_240','241_to_270','271_to_300']:
        return 'ANC'
    return adoption

df['Type_of_adoption'] = df.apply(type_of_adoption, axis=1)

# Simpler groupings
df['Adoption_type_RM_1'] = df['Mother adoption till birth'].apply(
    lambda x: 'PNC + ANC last month adoption' if x in ['PNC','001_to_030']
    else 'ANC' if pd.notna(x) else 'No value'
)
df['Adoption_type_RM_2'] = df['Mother adoption till birth'].apply(
    lambda x: 'PNC + ANC last two month adoption' if x in ['PNC','001_to_030','031_to_060']
    else 'ANC' if pd.notna(x) else 'No value'
)
df['Adoption_type_DS'] = df['Mother adoption till birth'].apply(
    lambda x: 'Purely PNC' if x == 'PNC'
    else 'Purely ANC' if pd.notna(x) else 'No value'
)

# ── Adoption_type_RM_3 (PNC split by age ≤150, within first 2 months + PNC) ──
def classify_rm3(row):
    adoption = row['Mother adoption till birth']
    age = row['Age of adoption']
    if adoption in ['PNC', '001_to_030', '031_to_060']:
        if adoption == 'PNC':
            if pd.notna(age) and 0 <= age <= 150:
                return 'PNCLTE5M + ANC last two month adoption'
            elif pd.notna(age) and age > 150:
                return 'PNCGT5M'
        return 'ANC'
    elif pd.notna(adoption):
        return 'ANC'
    return 'No value'

df['Adoption_type_RM_3'] = df.apply(classify_rm3, axis=1)

# ── Adoption_type_1 (PNC → PNCL5M/PNCG5M, ANC → <60D / >=60D) ──────────────
def classify_adoption_type_1(row):
    adoption = row['Mother adoption till birth']
    age = row['Age of adoption']
    if pd.isna(adoption) and pd.isna(age):
        return 'Both mother adoption and age of adoption missing'
    if pd.isna(adoption): return 'Mother Adoption missing'
    if pd.isna(age):      return 'Age of adoption missing'
    if adoption == 'PNC':
        if 0 <= age <= 150: return 'PNCL5M'
        if age > 150:       return 'PNCG5M'
    if adoption in ['001_to_030', '031_to_060']:
        return 'ANC<60D'
    return 'ANC>=60D'

df['Adoption_type_1'] = df.apply(classify_adoption_type_1, axis=1)

# ── ANC / BF / CF followup periods ───────────────────────────────────────────
def classify_anc_fu(val):
    if pd.isna(val) or val == 'PNC': return 'No Followup'
    if val in ['001_to_030','031_to_060','061_to_090']: return '<90 days'
    return '>=90 days' if str(val).startswith(('09','12','15','18','21','27','30')) else 'No Followup'

df['ANC_FU'] = df['Mother adoption till birth'].apply(classify_anc_fu)

def classify_bf_fu(row):
    if pd.isna(row['Baby Adoption date_CR']): return 'No Followup'
    age = row['Age of adoption']
    if pd.isna(age) or age > 150: return 'No Followup'
    dur = min(row['Age at Last Visit'], 150) - age
    return '<90 days' if dur < 90 else '>=90 days'


def classify_cf_fu(row):
    if pd.isna(row['Baby Adoption date_CR']): return 'No Followup'
    age = row['Age of adoption']
    if pd.isna(age) or age <= 150: return 'No Followup'
    dur = row['Age at Last Visit'] - max(age, 150)
    return '<90 days' if dur < 90 else '>=90 days'


df['BF_FU'] = df.apply(classify_bf_fu, axis=1)
df['CF_FU'] = df.apply(classify_cf_fu, axis=1)
df['combined_followup'] = list(zip(df['ANC_FU'], df['BF_FU'], df['CF_FU']))

print('Adoption type & follow-up classification columns created')

Adoption type & follow-up classification columns created


In [17]:
# Initialise blank exclusion reason columns
for col in ['Exclusion Reason_PNC','Exclusion Reason_ANC',
            'Exclusion Reason_ANC_2','Exclusion Reason_PNC_2']:
    df[col] = ''

pnc_mask  = df['Adoption_type_RM_1'] == 'PNC + ANC last month adoption'
anc_mask  = df['Adoption_type_RM_2'] == 'ANC'
anc2_mask = df['Adoption_type_RM_2'] == 'PNC + ANC last two month adoption'
pnc2_mask = df['Adoption_type_RM_2'].isin(['PNC + ANC last two month adoption','ANC'])

adoption_cols    = ['Visit Date_2','Weight_2','Height_2','Baby Adoption date_CR',
                    'Baby Adoption Weight_CR','Baby Adoption Weight Zscore_CR',
                    'Baby Adoption Percentile (W)_CR']
zscore_bv_av_lv_w = ['Last Weight Zscore_CR','Baby Adoption Weight Zscore_CR','Birth Weight Zscore_CR']
zscore_bv_av_lv_h = ['Last Height Zscore_CR','Baby Adoption Height Zscore_CR','Birth Height Zscore_CR']
zscore_bv_av_lv_f = ['Last WFH Zscore_CR','Baby Adoption WFH Zscore_CR','Birth WFH Zscore_CR']

# ── Generic helper to append exclusion text ──────────────────────────────────
def add_excl(mask, col, reason):
    df.loc[mask, col] += reason + '; '

# ── Common exclusions applied across all four sets ────────────────────────────
for mask, excl_col in [
    (pnc_mask,  'Exclusion Reason_PNC'),
    (anc_mask,  'Exclusion Reason_ANC'),
    (anc2_mask, 'Exclusion Reason_ANC_2'),
    (pnc2_mask, 'Exclusion Reason_PNC_2'),
]:
    add_excl(mask & df[['Child ID_all']].isnull().all(axis=1),
             excl_col, 'No Child ID - Child not born yet/child not followed up')
    add_excl(mask & df[BIRTH_DATE_COL].isnull(),
             excl_col, 'Birth date not available')
    add_excl(mask & ((df['Birth Weight Zscore_CR'] < -6) | (df['Birth Weight Zscore_CR'] > 6) |
                     df['birthweight_category_2'].isin(['Below_-6SD','Above_+6SD'])),
             excl_col, 'Birth weight category is below -6SD or above +6SD')
    add_excl(mask & df[zscore_bv_av_lv_w].isna().any(axis=1),
             excl_col, 'Weight Zscore at LV/AV/BV is missing')
    add_excl(mask & df[zscore_bv_av_lv_h].isna().any(axis=1),
             excl_col, 'Height Zscore at LV/AV/BV is missing')
    add_excl(mask & df[zscore_bv_av_lv_f].isna().any(axis=1),
             excl_col, 'WFH Zscore at LV/AV/BV is missing')

# ── Global exclusions (applied to ALL rows, no mask) ─────────────────────
for excl_col in ['Exclusion Reason_PNC', 'Exclusion Reason_ANC',
                  'Exclusion Reason_ANC_2', 'Exclusion Reason_PNC_2']:
    add_excl(df[['User Role','User Role_CR','Role_M']].isin(ROLES_TO_EXCLUDE).any(axis=1),
             excl_col, 'User role belongs to excluded category')
    add_excl(df['User Role'].isna() & df['User Role_CR'].isna() & df['Role_M'].isna(),
             excl_col, 'User Role is not available')
    add_excl(df['Type_of_adoption'] == 'Invalid difference: Baby Adoption date earlier than Mother adoption date',
             excl_col, 'Invalid difference: Baby Adoption date earlier than Mother adoption date')


# ── mother_AD_DOB > 300 applied GLOBALLY (no mask) ───────────────────────────
_mad_300 = (df['mother_AD_DOB'].apply(pd.to_numeric, errors='coerce') > 300).fillna(False)
for excl_col in ['Exclusion Reason_PNC', 'Exclusion Reason_ANC',
                  'Exclusion Reason_ANC_2', 'Exclusion Reason_PNC_2']:
    add_excl(_mad_300, excl_col, 'mother_AD_DOB > 300')

# ── PNC / PNC2-specific checks ────────────────────────────────────────────────
visit1_data_cols = ['Weight_1', 'Height_1', 'Visit Date_1']
visit2_data_cols = ['Weight_2', 'Height_2', 'Visit Date_2']

for mask, excl_col in [(pnc_mask, 'Exclusion Reason_PNC'), (pnc2_mask, 'Exclusion Reason_PNC_2')]:
    add_excl(mask & df[adoption_cols].isnull().all(axis=1),
             excl_col, 'Adoption visit anthropometry data(Wt,Ht,zscore) not available')
    
    # Only birth anthropometry details available: PNC_2 does not check visit data
    if excl_col == 'Exclusion Reason_PNC':
        add_excl(mask & (df['number_of_visits'] == 0) & df[visit1_data_cols].isna().all(axis=1),
                 excl_col, 'Only birth anthropometry details available')
    else:
        add_excl(mask & (df['number_of_visits'] == 0),
                 excl_col, 'Only birth anthropometry details available')
                 
    # Only adoption visit occurred: PNC_2 does not check visit data
    if excl_col == 'Exclusion Reason_PNC':
        add_excl(mask & (df['number_of_visits'] == 1) & df[visit2_data_cols].isna().all(axis=1),
                 excl_col, 'Only adoption visit occurred')
    else:
        add_excl(mask & (df['number_of_visits'] == 1),
                 excl_col, 'Only adoption visit occurred')
                 
    # Wt Gain per day >= 180 check: Writes strictly to 'Exclusion Reason_PNC' in both loops
    add_excl(mask & (df['avg_gain_category_LV_AV'] == 'More than 180'),
             'Exclusion Reason_PNC', 'Wt Gain per day ≥ 180gm between Adoption and Last visit')
             
    add_excl(mask & df['Birth Weight_CR'].notna() & df['Weight_1'].notna() &
             (df['Birth Weight_CR'] != df['Weight_1']),
             excl_col, 'Birth Weight in CR sheet and Visit 1 Weight in CM sheet do not match')
    add_excl(mask & df[BIRTH_DATE_COL].notna() & df['Visit Date_1'].notna() &
             (df[BIRTH_DATE_COL] != df['Visit Date_1']),
             excl_col, 'Date of birth in CR sheets and Visit Date 1 in CM sheet do not match')
    add_excl(mask & df['Visit Date_2'].notna() & df['Baby Adoption date_CR'].notna() &
             (df['Visit Date_2'] != df['Baby Adoption date_CR']),
             excl_col, 'Adoption date in CR sheet and Visit Date 2 in CM sheet do not match')
    add_excl(mask & df['Last Visit Date_CR'].notna() & df['Visit Date_last'].notna() &
             (df['Last Visit Date_CR'] != df['Visit Date_last']),
             excl_col, 'Last visit dates do not match in CR and CM sheets')
    add_excl(mask & df['Last Visit Date_CR'].isnull(),
             excl_col, 'Last visit Date not available')
    
    same_date = (
        df[BIRTH_DATE_COL].notna() & df['Baby Adoption date_CR'].notna() & df['Last Visit Date_CR'].notna() &
        (df[BIRTH_DATE_COL] == df['Baby Adoption date_CR']) &
        (df[BIRTH_DATE_COL] == df['Last Visit Date_CR'])
    )
    add_excl(mask & same_date, excl_col, 'Birth date, Adoption date, and Last visit date are the same')
    
    # Mother adoption date after last visit of the child: Writes strictly to Exclusion Reason_PNC_2 in both loops
    add_excl(mask & (df['adoption_duration_mother_child'] == 'negatives'),
             'Exclusion Reason_PNC_2', 'Mother adoption date is after last visit date of the child - suggestive of data entry error')

# ── ANC-specific checks ───────────────────────────────────────────────────────
for mask, excl_col in [(anc_mask, 'Exclusion Reason_ANC'), (anc2_mask, 'Exclusion Reason_ANC_2')]:
    add_excl(mask & df['Birth Weight_CR'].notna() & df['Weight_1'].notna() &
             (df['Birth Weight_CR'] != df['Weight_1']),
             excl_col, 'Birth Weight in CR sheet and Visit 1 Weight in CM sheet do not match')
    if excl_col == 'Exclusion Reason_ANC':
        add_excl(mask & df[BIRTH_DATE_COL].notna() & df['Visit Date_1'].notna() &
                  (df[BIRTH_DATE_COL] != df['Visit Date_1']),
                  excl_col, 'Date of birth in CR sheets and Visit Date 1 in CM sheet do not match')
    else:  # ANC_2
        add_excl(mask & df[BIRTH_DATE_COL].notna() & df['Visit Date_1'].notna() &
                  (df[BIRTH_DATE_COL] != df['Visit Date_1']),
                  excl_col, 'DOB in CR sheets and Visit Date 1 in CM sheet do not match')


# ── Replace blank with 'Included' ────────────────────────────────────────────
for col in ['Exclusion Reason_PNC','Exclusion Reason_ANC',
            'Exclusion Reason_ANC_2','Exclusion Reason_PNC_2']:
    df[col] = df[col].fillna('Included').replace(r'^\s*$', 'Included', regex=True)

# ── Primary exclusion (priority-based single reason) ─────────────────────────
PRIORITY_ORDER = [
    'No Child ID - Child not born yet/child not followed up',
    'Birth date not available',
    'Adoption visit anthropometry data(Wt,Ht,zscore) not available',
    'Mother adoption date is after last visit date of the child - suggestive of data entry error',
    'Birth weight category is below -6SD or above +6SD',
    'Birth Weight in CR sheet and Visit 1 Weight in CM sheet do not match',
    'Date of birth in CR sheets and Visit Date 1 in CM sheet do not match',
    'Adoption date in CR sheet and Visit Date 2 in CM sheet do not match',
    'Last visit dates do not match in CR and CM sheets',
    'Wt Gain per day ≥ 180gm between Adoption and Last visit',
    'Only adoption visit occurred',
    'Only birth anthropometry details available',
    'mother_AD_DOB > 300',
    'Birth date, Adoption date, and Last visit date are the same',
    'User role belongs to excluded category',
    'User Role is not available',
    'Weight Zscore at LV/AV/BV is missing',
    'Height Zscore at LV/AV/BV is missing',
    'WFH Zscore at LV/AV/BV is missing',
    'Invalid difference: Baby Adoption date earlier than Mother adoption date',
    'Last visit Date not available',
]

def assign_primary_exclusion(excl_col):
    primary = pd.Series([''] * len(df), index=df.index)
    for reason in PRIORITY_ORDER:
        mask = (primary == '') & df[excl_col].str.contains(reason, regex=False)
        primary[mask] = reason
    primary[primary == ''] = 'Included'
    return primary

df['Primary Exclusion_PNC']   = assign_primary_exclusion('Exclusion Reason_PNC')
df['Primary Exclusion_ANC']   = assign_primary_exclusion('Exclusion Reason_ANC')
df['Primary Exclusion_ANC_2'] = assign_primary_exclusion('Exclusion Reason_ANC_2')
df['Primary Exclusion_PNC_2'] = assign_primary_exclusion('Exclusion Reason_PNC_2')

df['Included_excluded'] = np.where(
    df['Primary Exclusion_PNC_2'].isin(['', 'Included']), 'Included', 'Excluded'
)

print('Exclusion logic applied')
print(df['Included_excluded'].value_counts().to_string())

Exclusion logic applied
Included_excluded
Included    1017
Excluded     952


In [18]:
# ── PNC visit presence flags ──────────────────────────────────────────────────
age_visit_cols_clean = [c for c in df.columns if c.startswith('baby_age_visit_')]

df['is_ANC_case']         = df['Mother Adoption date_CR'] < df[BIRTH_DATE_COL]
df['last_visit_equals_birth'] = df['Last Visit Date_CR'] == df[BIRTH_DATE_COL]
df['last_visit_after_birth']  = df['Last Visit Date_CR'] > df[BIRTH_DATE_COL]
df['PNC_0_150_visit']     = df[age_visit_cols_clean].apply(
    lambda x: ((x >= 0) & (x <= 150)).any(), axis=1
)

def assign_followup_category(row):
    for chk, lbl in [
        (pd.isna(row[BIRTH_DATE_COL]),               'Unclassified: Missing DOB'),
        (pd.isna(row['Mother Adoption date_CR']),     'Unclassified: Missing Adoption Date'),
        (pd.isna(row['Age at Last Visit']),           'Unclassified: Missing Last Visit Age'),
        (pd.isna(row['Age of adoption']),             'Unclassified: Missing BAOA'),
    ]:
        if chk: return lbl

    if row['is_ANC_case']:
        if row['Last Visit Date_CR'] < row[BIRTH_DATE_COL]:  return 'Data Issue: Last visit before birth'
        if row['last_visit_equals_birth']:                    return 'Category 5 : ANC Only'
        if row['last_visit_after_birth']:
            lv_age = row['Age at Last Visit']
            if 0 <= lv_age <= 150: return 'Category 2 : ANC + PNC<5M'
            if lv_age > 150:
                return 'Category 1 : ANC + PNC<5 + PNC>5' if row['PNC_0_150_visit'] \
                       else 'Category 4 : ANC + PNC>5'
    else:
        baoa = row['Age of adoption']
        if baoa < 0: return 'Data Issue: Negative BAOA'
        if baoa > 150: return 'Category 7 : PNC>5M Only'
        lv_age = row['Age at Last Visit']
        if pd.isna(lv_age): return 'Unclassified: Missing Last Visit Age'
        if 0 <= lv_age <= 150: return 'Category 6 : PNC<5M Only'
        if lv_age > 150:       return 'Category 3 : PNC<5 + PNC>5'
    return 'Unclassified'

df['Followup Category'] = df.apply(assign_followup_category, axis=1)

# ── ANC / PNC / Protein visit counts ─────────────────────────────────────────
anc_visit_cols    = [c for c in df.columns if c.startswith('Assessment_date_') and c.endswith('_An')]
pnc_visit_cols_v  = [c for c in df.columns
                     if c.startswith('Visit Date_') and '-' not in c]
protein_date_cols = [c for c in df.columns if '_P' in c and 'Assessment_date' in c and '-' not in c]

for col in anc_visit_cols + pnc_visit_cols_v + protein_date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')

def count_anc_visits(row):
    s, e = row['Mother Adoption date_CR'], row[BIRTH_DATE_COL]
    if pd.isna(s) or pd.isna(e) or s > e: return 0
    return sum(1 for c in anc_visit_cols if pd.notna(row[c]) and s <= row[c] <= e)

def count_pnc_lt5(row):
    dob, aoa = row[BIRTH_DATE_COL], row['Age of adoption']
    if pd.isna(dob) or pd.isna(aoa): return 0
    return sum(1 for c in pnc_visit_cols_v
               if pd.notna(row[c]) and aoa <= (row[c] - dob).days <= 150)

def count_pnc_gt5(row):
    dob = row[BIRTH_DATE_COL]
    if pd.isna(dob): return 0
    return sum(1 for c in pnc_visit_cols_v if pd.notna(row[c]) and (row[c] - dob).days > 150)

def count_protein(row, is_anc):
    adm, dob = row['Mother Adoption date_CR'], row[BIRTH_DATE_COL]
    if pd.isna(adm) or pd.isna(dob): return 0
    return sum(1 for c in protein_date_cols if pd.notna(row[c]) and
               (adm <= row[c] < dob if is_anc else row[c] >= dob))

df['ANC_visit_count']    = df.apply(count_anc_visits, axis=1)
df['PNC_lt5_visit_count']= df.apply(count_pnc_lt5, axis=1)
df['PNC_gt5_visit_count']= df.apply(count_pnc_gt5, axis=1)
df['ANC_protein_count']  = df.apply(lambda r: count_protein(r, is_anc=True), axis=1)
df['PNC_protein_count']  = df.apply(lambda r: count_protein(r, is_anc=False), axis=1)

def debug_protein_dates(row):
    mother_adopt = row['Mother Adoption date_CR']
    dob = row[BIRTH_DATE_COL]
    anc_dates, pnc_dates = [], []
    if pd.notna(mother_adopt) and pd.notna(dob):
        for col in protein_date_cols:
            visit = row[col]
            if pd.isna(visit): continue
            if mother_adopt <= visit < dob:
                anc_dates.append(visit)
            elif visit >= dob:
                pnc_dates.append(visit)
    return pd.Series([anc_dates, pnc_dates])

df[['ANC_dates_debug', 'PNC_dates_debug']] = df.apply(debug_protein_dates, axis=1)
df['total_protein_dates'] = df[protein_date_cols].notna().sum(axis=1)
df['check_total'] = df['ANC_protein_count'] + df['PNC_protein_count']

# ── Protein reference note (diagnostic) ──────────────────────────────────────
def protein_ref_note(row):
    adm = row['Mother Adoption date_CR']
    dob = row[BIRTH_DATE_COL]
    if pd.isna(adm) and pd.isna(dob):
        return 'Both Mother Adoption date and Birth date missing -> Cannot classify ANC/PNC.'
    if pd.isna(adm):
        return 'Mother Adoption date missing -> Cannot determine ANC boundary.'
    if pd.isna(dob):
        return 'Birth date missing -> Cannot determine PNC boundary.'
    return 'Used both Mother Adoption date and Birth date.'

df['Protein_reference_note'] = df.apply(protein_ref_note, axis=1)

# Duration columns
df['ANC_duration']     = (df[BIRTH_DATE_COL] - df['Mother Adoption date_CR']).dt.days.clip(lower=0)
df['PNC_lt5_duration'] = (150 - df['Age of adoption']).clip(lower=0)
df['PNC_gt5_duration'] = (df['Age at Last Visit'] - 150).clip(lower=0)

# Adequacy flags
df['ANC_60d_3visit']    = df.apply(
    lambda r: 'None' if pd.isna(r['ANC_duration']) else ('Yes' if r['ANC_duration'] >= 60 and r['ANC_visit_count'] >= 3 else 'No'), axis=1)
df['PNC_lt5_60d_8visit']= df.apply(
    lambda r: 'None' if pd.isna(r['PNC_lt5_duration']) else ('Yes' if r['PNC_lt5_duration'] >= 60 and r['PNC_lt5_visit_count'] >= 8 else 'No'), axis=1)
df['PNC_gt5_60d_4visit']= df.apply(
    lambda r: 'None' if pd.isna(r['PNC_gt5_duration']) else ('Yes' if r['PNC_gt5_duration'] >= 60 and r['PNC_gt5_visit_count'] >= 4 else 'No'), axis=1)

# Protein categories
visit_bins = [-1, 0, 1, 3, 5, 7, 9, float('inf')]
visit_lbls = ['00_visit','01_visits','02_to_03_visits','04_to_05_visits',
              '06_to_07_visits','08_to_09_visits','10_or_more_visits']
df['ANC_protein_category'] = pd.cut(df['ANC_protein_count'], bins=visit_bins, labels=visit_lbls, right=True)
df['PNC_protein_category'] = pd.cut(df['PNC_protein_count'], bins=visit_bins, labels=visit_lbls, right=True)

# Mark unclassified followup in exclusion columns
excl_cols_all = ['Exclusion Reason_PNC_2','Exclusion Reason_PNC',
                 'Exclusion Reason_ANC','Exclusion Reason_ANC_2',
                 'Primary Exclusion_PNC_2','Primary Exclusion_PNC',
                 'Primary Exclusion_ANC','Primary Exclusion_ANC_2']
unc_mask = df['Followup Category'] == 'Unclassified'
df.loc[unc_mask, excl_cols_all] = 'Follow-up category is Unclassified '

print('Follow-up category & visit count columns created')
print(df['Followup Category'].value_counts().to_string())

Follow-up category & visit count columns created
Followup Category
Category 6 : PNC<5M Only                564
Unclassified: Missing DOB               411
Category 7 : PNC>5M Only                282
Category 3 : PNC<5 + PNC>5              254
Category 2 : ANC + PNC<5M               206
Unclassified: Missing BAOA              202
Unclassified: Missing Last Visit Age     33
Category 1 : ANC + PNC<5 + PNC>5         15
Unclassified: Missing Adoption Date       1
Category 5 : ANC Only                     1


In [19]:
# ── Number of adoptions after exclusion ───────────────────────────────────────
included_mask = df['Exclusion Reason_PNC_2'] == 'Included'
df['number_of_adoption_after_exclusion'] = np.nan
df.loc[included_mask, 'number_of_adoption_after_exclusion'] = (
    df[included_mask].groupby('User Acc ID_all')['User Acc ID_all'].transform('count')
)
df['Total_Adoptions_group_after_exclusion'] = df['number_of_adoption_after_exclusion'].apply(categorize_adoption_count)
df['Total_Adoptions_group_after_exclusion_jalna'] = df['number_of_adoption_after_exclusion'].apply(categorize_adoption_count_jalna)
df['diff_btw_adoption_count_after_exclusion'] = df['number_of_adoption'] - df['number_of_adoption_after_exclusion']

# ── Impact % after exclusion ──────────────────────────────────────────────────
for cnt, pct in [
    ('impact_child_count_RD',   'Total_impact_children_for_learner_%_RD_after_exclusion'),
    ('impact_child_count_RD_2', 'Total_impact_children_for_learner_%_RD_2_after_exclusion'),
    ('impact_child_count_DP',   'Total_impact_children_for_learner_%_DP_after_exclusion'),
]:
    df[pct] = (df[cnt] / df['number_of_adoption_after_exclusion'] * 100).round(1)

for suffix in ['RD', 'RD_2', 'DP']:
    df[f'impact_children_%_{suffix}_group_after_exclusion'] = pd.cut(
        df[f'Total_impact_children_for_learner_%_{suffix}_after_exclusion'],
        bins=pct_bins, labels=pct_lbls, include_lowest=True
    )

# ── Learner: WFA Normal proportion ───────────────────────────────────────────
df['is_normal'] = (df['WFA_status_at_LV'] == 'Normal').astype(int)
ls = df.groupby('User Acc ID_all').agg(
    normal_adoptions=('is_normal','sum'),
    _noa=('number_of_adoption','first')
).assign(normal_pct=lambda x: (x['normal_adoptions'] / x['_noa'] * 100).round(2)).reset_index()
ls['Learner_Normal_60pct'] = (ls['normal_pct'] >= 60).map({True:'Yes', False:'No'})
ls['Learner_Normal_Range'] = pd.cut(ls['normal_pct'], bins=[0,20,40,60,80,101],
                                    labels=['0-19','20-39','40-59','60-79','80-100'], right=False)
df = df.merge(ls[['User Acc ID_all','normal_adoptions','normal_pct',
                  'Learner_Normal_60pct','Learner_Normal_Range']],
              on='User Acc ID_all', how='left')

# ── Learner: WFA improvement ──────────────────────────────────────────────────
df['Improved_WFA_Zscore'] = (df['Last_visit_zscore_weight - adoption_zscore'] > 0).astype(int)
ls2 = df.groupby('User Acc ID_all').agg(
    improved_cases=('Improved_WFA_Zscore','sum'),
    _noa=('number_of_adoption','first')
).assign(Improvement_pct=lambda x: (x['improved_cases'] / x['_noa'] * 100)).reset_index()
ls2['Improvement in WFA Zscore btw AV_LV'] = (ls2['Improvement_pct'] >= 60).map({True:'Yes', False:'No'})
ls2['Learner_Improvement_Range'] = pd.cut(ls2['Improvement_pct'],
    bins=[0,20,40,60,80,101], labels=['0-19','20-39','40-59','60-79','80-100'], right=False)
df = df.merge(ls2[['User Acc ID_all','improved_cases','Improvement_pct',
                   'Improvement in WFA Zscore btw AV_LV','Learner_Improvement_Range',
                   'improved_cases']],
              on='User Acc ID_all', how='left')

# ── Learner: Catchup proportion ───────────────────────────────────────────────
df['Catchup_flag'] = (df['weight_zscore_change_group_btw_LV_AV'] == 'Catchup').astype(int)
ls3 = df.groupby('User Acc ID_all').agg(
    catchup_cases=('Catchup_flag','sum'),
    _noa=('number_of_adoption','first')
).assign(Catchup_pct=lambda x: (x['catchup_cases'] / x['_noa'] * 100)).reset_index()
ls3['Catchup_YN'] = (ls3['Catchup_pct'] >= 60).map({True:'Yes', False:'No'})
ls3['Catchup_pct_range'] = pd.cut(ls3['Catchup_pct'],
    bins=[0,20,40,60,80,101], labels=['0-19','20-39','40-59','60-79','80-100'], right=False)
df = df.merge(ls3[['User Acc ID_all','catchup_cases','Catchup_YN','Catchup_pct','Catchup_pct_range']],
              on='User Acc ID_all', how='left')

# ── Learner: avg wt gain >17g and >28g proportions ────────────────────────────
for threshold, col_sfx in [(17, '17g'), (28, '28g')]:
    df[f'avg_wt_gain_{col_sfx}_flag'] = (df['Avg_Weight_Gain_per_Day_LV_AV'] > threshold).astype(int)
    ls_wg = df.groupby('User Acc ID_all').agg(
        **{f'avg_wt_gain_{col_sfx}_cases': (f'avg_wt_gain_{col_sfx}_flag', 'sum')},
        _noa=('number_of_adoption','first')
    ).assign(**{f'avg_wt_gain_{col_sfx}_pct':
               lambda x, c=col_sfx: (x[f'avg_wt_gain_{c}_cases'] / x['_noa'] * 100)}).reset_index()
    ls_wg[f'avg_weight_gain_{col_sfx}_60pct_YN'] = (
        ls_wg[f'avg_wt_gain_{col_sfx}_pct'] >= 60).map({True:'Yes', False:'No'})
    ls_wg[f'avg_wt_gain_{col_sfx}_pct_range'] = pd.cut(
        ls_wg[f'avg_wt_gain_{col_sfx}_pct'],
        bins=[0,20,40,60,80,101], labels=['0-19','20-39','40-59','60-79','80-100'], right=False)
    df = df.merge(ls_wg[['User Acc ID_all', f'avg_wt_gain_{col_sfx}_cases',
                          f'avg_wt_gain_{col_sfx}_pct', f'avg_weight_gain_{col_sfx}_60pct_YN',
                          f'avg_wt_gain_{col_sfx}_pct_range']],
                  on='User Acc ID_all', how='left')

# ── Fine-grained avg wt gain category ────────────────────────────────────────
bins_2 = [-np.inf, 0, 5, 10, 15, 17, 20, 25, 28, 30, 35, 40, np.inf]
lbls_2 = ['0 or less','0.1–5 g/d','5.1–10 g/d','10.1–15 g/d','15.1–17 g/d',
          '17.1–20 g/d','20.1–25 g/d','25.1–28 g/d','28.1–30 g/d',
          '30–35 g/d','35.1–40 g/d','More than 40 g/d']
df['Avg_Weight_Gain_Category_LV_AV_2'] = pd.cut(
    df['Avg_Weight_Gain_per_Day_LV_AV'], bins=bins_2, labels=lbls_2
)

print('Learner-level aggregation columns created')
print(f'Final shape: {df.shape}')

Learner-level aggregation columns created
Final shape: (1969, 3487)


In [ ]:
# ==============================================================================
# BLOCKS + ANALYSIS FLAGS ADD-ON (requested 29-07-2026, append-only)
# ------------------------------------------------------------------------------
# The block variable used to be built inside the crosstab stage, so the derived
# sheet carried no block at all and any plot made from it could not line up with
# the crosstabs. It is now built HERE, once, and the crosstab stage simply uses
# the column this cell writes -- one source of truth, so the two can never drift.
#
# Adds three columns, and REMOVES NOTHING (every row is kept for audit):
#   'Blocks'            - the learner's block: from 'Phc Taluka_CR', normalised
#                         (case, trailing '-1'/'-2', spelling variants merged),
#                         then gap-filled from MASD and then from the one-time
#                         DOCS sheet. 'z_NA' where still unknown.
#   'In analysis'       - 'Yes' if the row is one of the rows the crosstabs use,
#                         'No' otherwise. Filter on Yes to reproduce the tables.
#   'Excluded because'  - why a row is not in the analysis (blank when it is).
#
# The exclusion rules below MUST stay identical to ADD-ON DATA STEP 3 in the
# crosstab notebook. If you change one, change the other.
# ==============================================================================
import os as _os_blk, glob as _glob_blk, re as _re_blk

DERIVED_BLOCK_COL = 'Blocks'
BLOCK_NAME_ALIASES = {          # spelling variants / merges of the SAME block
    'Ghatia': 'Ghatiya',
    'Khachrod': 'Khacharod',
    # 03-08-2026 (project lead): the rural sub-area is reported as part of
    # Ujjain block, so its cases and learners fold into 'Ujjain' everywhere --
    # derived sheet and every crosstab, row-wise and column-wise alike.
    'Ujjain - Rural': 'Ujjain',
    'Ujjain-Rural': 'Ujjain',
    'Ujjain Rural': 'Ujjain',
}
EXCLUDED_ROLE_GROUPS_ML = ['Nursing staff']   # must match the crosstab notebook


def _normalise_block(v):
    if pd.isna(v):
        return 'z_NA'
    s = str(v).strip()
    if s == 'z_NA':                                  # already normalised
        return s
    s = _re_blk.sub(r'[\s\-_]*\d+$', '', s)          # strip trailing '-1' / '-2'
    s = _re_blk.sub(r'\s+', ' ', s).strip().title()
    s = BLOCK_NAME_ALIASES.get(s, s)                 # merge spelling variants
    return s if s else 'z_NA'


_blk_id = 'User Acc ID_all' if 'User Acc ID_all' in df.columns else (
    'User Acc ID all' if 'User Acc ID all' in df.columns else None)
_blk_src = 'Phc Taluka_CR' if 'Phc Taluka_CR' in df.columns else None

if _blk_src:
    df[DERIVED_BLOCK_COL] = df[_blk_src].apply(_normalise_block)
else:
    df[DERIVED_BLOCK_COL] = 'z_NA'
print(f"  Derived '{DERIVED_BLOCK_COL}' from '{_blk_src}': "
      f"{dict(df[DERIVED_BLOCK_COL].value_counts())}")

_blk_dir = _os_dyn.path.dirname(_os_dyn.path.abspath(str(ROLE_SHEET)))


def _blk_fill(lut, label):
    """Gap-fill Blocks from {learner id -> block}; never overwrite, never add a
    block label the case data does not already use."""
    if not lut or not _blk_id:
        return 0, 0
    miss = df[DERIVED_BLOCK_COL].astype(str).eq('z_NA')
    if not miss.any():
        return 0, 0
    n0, l0 = int(miss.sum()), df.loc[miss, _blk_id].astype(str).nunique()
    known = set(df.loc[~miss, DERIVED_BLOCK_COL].astype(str).unique())
    found = (df.loc[miss, _blk_id].astype(str).str.strip()
             .str.replace(r'\.0$', '', regex=True).map(lut))
    rej = found[found.notna() & ~found.isin(known)]
    if len(rej):
        print(f"  {label}: REJECTED {len(rej)} row(s) proposing an unknown block"
              f" label {dict(rej.value_counts())}")
        found = found.where(found.isin(known))
    hit = found.notna()
    df.loc[found[hit].index, DERIVED_BLOCK_COL] = found[hit]
    left = df[DERIVED_BLOCK_COL].astype(str).eq('z_NA')
    n1, l1 = int(left.sum()), df.loc[left, _blk_id].astype(str).nunique()
    print(f"  {label}: filled {n0 - n1} of {n0} missing case row(s)"
          f" ({l0 - l1} of {l0} learner(s) resolved)")
    return n0 - n1, l0 - l1


# ---- fill 1: the MASD combined sheet ----------------------------------------
_masd_hits = ([] if not HELPER_DISTRICT else _glob_blk.glob(
    _os_dyn.path.join(_blk_dir, f'MASD_{HELPER_DISTRICT}_*_combined.xlsx')))
_masd_hits = [h for h in _masd_hits if not _os_dyn.path.basename(h).startswith('~$')]
if _masd_hits:
    _mp = max(_masd_hits, key=_os_dyn.path.getmtime)
    _md = pd.read_excel(_mp)
    if 'User Account ID' in _md.columns and 'Block/Taluk' in _md.columns:
        _blk_fill({str(k).strip(): _normalise_block(v)
                   for k, v in zip(_md['User Account ID'], _md['Block/Taluk'])
                   if isinstance(v, str) and v.strip()},
                  f'MASD block fill ({_os_dyn.path.basename(_mp)})')
    else:
        print('  MASD sheet lacks the ID / Block-Taluk columns -> block fill skipped')
else:
    print('  MASD combined sheet not found -> MASD block fill skipped')

# ---- fill 2: the one-time DOCS sheet ----------------------------------------
_docs_hits = ([] if not HELPER_DISTRICT else _glob_blk.glob(
    _os_dyn.path.join(_blk_dir, f'DOCS_onetime_fill_{HELPER_DISTRICT}.xlsx')))
if _docs_hits:
    _dp = max(_docs_hits, key=_os_dyn.path.getmtime)
    _dd = pd.read_excel(_dp)
    if 'User Acc ID' in _dd.columns and 'Block' in _dd.columns:
        _blk_fill({str(k).strip(): _normalise_block(v)
                   for k, v in zip(_dd['User Acc ID'], _dd['Block'])
                   if isinstance(v, str) and v.strip()},
                  f'DOCS block fill ({_os_dyn.path.basename(_dp)})')
else:
    print('  DOCS one-time sheet not found -> DOCS block fill skipped (normal for a new dataset)')

print(f"  Final '{DERIVED_BLOCK_COL}': {dict(df[DERIVED_BLOCK_COL].value_counts())}")

# ---- REASON FOR PRE-ANALYSIS REMOVAL (02-08-2026) ----------------------------
# Requested by the project lead: every case row carries WHY it is excluded from
# the crosstab analysis, in ONE column, assigned in this fixed order (first
# matching reason wins; blank = the row IS in the analysis):
#   1. HST trainer team              raw role matches the HST patterns
#   2. No training batch             learner absent from the MASD sheet, or
#                                    present with a blank 'Training Batch'
#                                    (MASD is the batch authority)
#   3. No role / role group defined  still no role after the DOCS + MASD fills
#   4. No block defined              still no block after the MASD + DOCS fills
#   5. Staff nurse role group        role group 'Nursing staff'
#   6. Zero adoptions in MASD        MASD 'Total adoptions' = 0 for the learner
# The crosstab stage removes every row with a non-blank reason, so this column
# exactly defines the crosstab analysis set ('Included' = the row IS in the
# analysis). The step-by-step numbers go into the 'Removal flowchart' sheet of
# this workbook. Column name per the project lead (02-08-2026):
REASON_COL_NAME = 'pre_analysis_exclusion'
R1 = 'HST trainer team (HST / calling-team role)'
R2 = 'No training batch (learner not in MASD or batch blank there)'
R3 = 'No role / role group defined (not resolvable from MASD or DOCS)'
R4 = 'No block defined (not resolvable from MASD or DOCS)'
R5 = 'Staff nurse role group (Nursing staff)'
R6 = 'Zero adoptions in MASD'
EXCLUDED_ROLE_GROUPS_ML = ['Nursing staff']   # must match the crosstab notebook

# batch + adoptions authority: the full MASD extract
import glob as _glob_rs
_rs_dir = _os_dyn.path.dirname(_os_dyn.path.abspath(str(ROLE_SHEET)))
_rs_hits = []
for _rs_pat in ([f'{HELPER_CODE}_raw_combined_*_MASD.xlsx'] if HELPER_CODE else []) \
        + ([f'MASD_{HELPER_DISTRICT}_*_combined.xlsx'] if HELPER_DISTRICT else []):
    for _rs_d in (_rs_dir, _os_dyn.path.dirname(_rs_dir)):
        _rs_hits += _glob_rs.glob(_os_dyn.path.join(_rs_d, _rs_pat))
    if _rs_hits:
        break
_rs_hits = [h for h in _rs_hits if not _os_dyn.path.basename(h).startswith('~$')]

_rs_in_masd, _rs_batched, _rs_adopt0 = set(), set(), set()
_HAVE_MASD = bool(_rs_hits)
if _HAVE_MASD:
    _rs_path = max(_rs_hits, key=_os_dyn.path.getmtime)
    _rs = pd.read_excel(_rs_path)
    _rs_ids = _rs['User Account ID'].astype(str).str.strip().str.replace(r'\.0$', '', regex=True)
    _rs_in_masd = set(_rs_ids)
    _rs_batched = {i for i, b in zip(_rs_ids, _rs['Training Batch'])
                   if isinstance(b, str) and b.strip()}
    _rs_adopt0 = {i for i, a in zip(_rs_ids, _rs['Total adoptions'])
                  if not pd.isna(a) and a == 0}
    print(f'  Reason engine: MASD authority = {_os_dyn.path.basename(_rs_path)}'
          f' ({len(_rs_in_masd)} learners, {len(_rs_in_masd) - len(_rs_batched)} without a'
          f' Training Batch, {len(_rs_adopt0)} with 0 adoptions)')
else:
    print('  !! MASD sheet not found -> reasons 2 (no batch) and 6 (0 adoptions)'
          ' CANNOT be applied and are skipped')

_rs_id = (df[_blk_id].astype(str).str.strip().str.replace(r'\.0$', '', regex=True)
          if _blk_id else pd.Series('', index=df.index))

_rs_hstpat = '|'.join([r'\bHST\b', r'calling\s*team'])
_rm1 = pd.Series(False, index=df.index)
for _hc in [c for c in ['User Role', 'User Role_CR', 'Role_M'] if c in df.columns]:
    _rm1 |= df[_hc].astype(str).str.contains(_rs_hstpat, case=False, na=False, regex=True)

_rm2 = (~_rs_id.isin(_rs_in_masd) | _rs_id.isin(_rs_in_masd - _rs_batched)) \
    if _HAVE_MASD else pd.Series(False, index=df.index)

_rg_s = df['Role Group'].astype(str).str.strip()
_rg_n = _rg_s.str.lower().str.replace('_', ' ', regex=False).str.split().str.join(' ')
_rm3 = df['Role Group'].isna() | _rg_n.isin(['z na', 'zna', 'nan', 'na', 'none', ''])
_rm4 = df[DERIVED_BLOCK_COL].astype(str).eq('z_NA')
_rm5 = _rg_s.str.lower().isin([g.strip().lower() for g in EXCLUDED_ROLE_GROUPS_ML])
_rm6 = _rs_id.isin(_rs_adopt0) if _HAVE_MASD else pd.Series(False, index=df.index)

df[REASON_COL_NAME] = 'Included'
_taken = pd.Series(False, index=df.index)
for _mask, _lab in [(_rm1, R1), (_rm2, R2), (_rm3, R3), (_rm4, R4), (_rm5, R5), (_rm6, R6)]:
    _pick = _mask & ~_taken
    df.loc[_pick, REASON_COL_NAME] = _lab
    _taken |= _pick

# ---- Removal flowchart (sequential, same order) -------------------------------
_fc_rows = []
_fc = df.copy()


def _fc_l(frame):
    return frame[_blk_id].astype(str).nunique() if _blk_id else len(frame)


_fc_rows.append({'Step': 'START: all cases in the derived sheet',
                 'Cases before': len(_fc), 'Learners before': _fc_l(_fc),
                 'Cases removed': '', 'Learners removed (left entirely)': '',
                 'Cases after': len(_fc), 'Learners after': _fc_l(_fc)})
for _lab in [R1, R2, R3, R4, R5, R6]:
    _b_c, _b_l = len(_fc), _fc_l(_fc)
    _keep = _fc[_fc[REASON_COL_NAME].ne(_lab)]
    _fc_rows.append({'Step': _lab,
                     'Cases before': _b_c, 'Learners before': _b_l,
                     'Cases removed': _b_c - len(_keep),
                     'Learners removed (left entirely)': _b_l - _fc_l(_keep),
                     'Cases after': len(_keep), 'Learners after': _fc_l(_keep)})
    _fc = _keep
_fc_rows.append({'Step': 'FINAL: used in the crosstab analysis',
                 'Cases before': '', 'Learners before': '',
                 'Cases removed': '', 'Learners removed (left entirely)': '',
                 'Cases after': len(_fc), 'Learners after': _fc_l(_fc)})
FLOWCHART_DF = pd.DataFrame(_fc_rows)

print(f"  {REASON_COL_NAME}: "
      f"{dict(df.loc[df[REASON_COL_NAME].ne('Included'), REASON_COL_NAME].value_counts())}")
_incl = df[REASON_COL_NAME].eq('Included')
print(f"  Included (in analysis): {int(_incl.sum())} of {len(df)} case row(s), "
      f"{_fc_l(df[_incl])} learner(s)")
print('  Removal flowchart:')
print(FLOWCHART_DF.to_string(index=False))
print("  NOTE: every row is kept in the derived sheet. Filter"
      " pre_analysis_exclusion = 'Included' to reproduce the crosstab numbers exactly.")


In [20]:
# ── Rename columns: replace '_' with ' ' (preserve module suffixes like _M, _P, _CR) ─

def clean_col_name(name):
    """Replace underscores with spaces, but keep module suffixes like _M, _P, _CR, _BF, etc."""
    # Module suffixes to preserve (check longer ones first)
    suffixes_to_preserve = [
        '_CR', '_BF', '_CF', '_CG', '_An',
        '_M', '_P', '_C',
    ]
    # Extract and temporarily remove trailing module suffixes
    preserved = ''
    temp = name
    while True:
        matched = False
        for sfx in suffixes_to_preserve:
            if temp.endswith(sfx):
                preserved = sfx + preserved
                temp = temp[:-len(sfx)]
                matched = True
                break
        if not matched:
            break
    # Replace all remaining underscores with spaces
    temp = temp.replace('_', ' ')
    return temp + preserved

# 'pre_analysis_exclusion' keeps its underscores by request (02-08-2026) --
# clean_col_name would turn it into 'pre analysis exclusion'
df.columns = [c if c == 'pre_analysis_exclusion' else clean_col_name(c) for c in df.columns]

# ── Write to Excel with formatting ──────────────────────────────────────────
from openpyxl.styles import Alignment

with pd.ExcelWriter(OUTPUT_FILE, engine='openpyxl') as writer:
    df.to_excel(writer, index=False, sheet_name='Sheet1')
    ws = writer.sheets['Sheet1']

    # Format header row: center-align + wrap text
    for cell in ws[1]:
        cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)

    # Freeze the header row
    ws.freeze_panes = 'A2'

    # ---- 'Removal flowchart' sheet (02-08-2026): the step-by-step reduction ----
    if 'FLOWCHART_DF' in dir() and FLOWCHART_DF is not None:
        FLOWCHART_DF.to_excel(writer, sheet_name='Removal flowchart', index=False)
        wsf = writer.sheets['Removal flowchart']
        for _w, _c in zip((64, 13, 15, 13, 18, 12, 14), 'ABCDEFG'):
            wsf.column_dimensions[_c].width = _w
        for cell in wsf[1]:
            cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
        wsf.freeze_panes = 'A2'

print(f'  Saved -> {OUTPUT_FILE}  |  Shape: {df.shape}')


# Quick sanity check
included = df[df['Exclusion Reason PNC 2'] == 'Included']
print(f'Included rows : {len(included)}')
print(f'Excluded rows : {len(df) - len(included)}')
print('\nFollow-up category breakdown:')
print(df['Followup Category'].value_counts().to_string())

  Saved -> UJ_220726_Exc_V(220726)_2.xlsx  |  Shape: (1969, 3487)
Included rows : 1017
Excluded rows : 952

Follow-up category breakdown:
Followup Category
Category 6 : PNC<5M Only                564
Unclassified: Missing DOB               411
Category 7 : PNC>5M Only                282
Category 3 : PNC<5 + PNC>5              254
Category 2 : ANC + PNC<5M               206
Unclassified: Missing BAOA              202
Unclassified: Missing Last Visit Age     33
Category 1 : ANC + PNC<5 + PNC>5         15
Unclassified: Missing Adoption Date       1
Category 5 : ANC Only                     1
